In [ ]:
# === kyouyuuyou (even): compute only (5-step + bootstrap CI) ===fig4 compute
import numpy as np
import pandas as pd
from pathlib import Path

# =========================
# Settings (edit here)
# =========================
MAX_GROUP_SIZE = 100
GROUP_STRIDE = 1
GROUP_SIZES = [1] + list(range(GROUP_STRIDE, MAX_GROUP_SIZE + 1, GROUP_STRIDE))
GROUP_SIZES = sorted({int(s) for s in GROUP_SIZES if 1 <= int(s) <= int(MAX_GROUP_SIZE)})

N_BOOT = 1500
N_INST = 1500
CI_ALPHA = 0.05  # 95% CI
SEED = 42

# Keep these for plot cell defaults
ERRORBAR_CAPSIZE = 4.0
ERRORBAR_ALPHA = 0.9
CI_BAND_ALPHA = 0.18

CACHE_DIR = Path('data/_cache_rrci')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Dataset
# =========================
CSV_RANDOM_PATH = Path('../../data/dr_even_base10000_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv')

if not CSV_RANDOM_PATH.exists():
    candidates = [
        Path('data') / 'density_random_custom.csv',
        Path('data') / 'density_random.csv',
        Path('../data/density_random_custom.csv'),
    ]
    for cand in candidates:
        if cand.exists():
            CSV_RANDOM_PATH = cand
            break
    else:
        raise FileNotFoundError('custom density_random CSV not found')

print('Using dataset:', CSV_RANDOM_PATH)
df_random = pd.read_csv(CSV_RANDOM_PATH)

# =========================
# Helpers
# =========================
def bootstrap_ci_mean(values, n_boot=2000, alpha=0.05, seed=42):
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[b] = arr[idx].mean()

    lo = np.percentile(boot_means, 100.0 * (alpha / 2.0))
    hi = np.percentile(boot_means, 100.0 * (1.0 - alpha / 2.0))
    return float(arr.mean()), float(lo), float(hi)


def _ensure_components(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    if 'zz_time_total_s' not in d.columns:
        if 'zz_time_total_ns' in d.columns:
            d['zz_time_total_s'] = d['zz_time_total_ns'].astype(float) / 1e9
        elif 'zz_time' in d.columns:
            d['zz_time_total_s'] = d['zz_time'].astype(float) / 1e9
        elif 'total_time' in d.columns:
            d['zz_time_total_s'] = d['total_time'].astype(float) / 1e9
        else:
            raise KeyError('Cannot derive zz_time_total_s')

    d['t_distance_per_shot_s'] = 5.0 * d['distance'].astype(float) / 200000.0

    if (d['entanglement_speed_factor'].astype(float) == 0).any():
        raise ZeroDivisionError('entanglement_speed_factor must be non-zero.')
    d['t_entangle_per_shot_s'] = 5.0 / d['entanglement_speed_factor'].astype(float)

    if (d['gate_speed_factor'].astype(float) == 0).any():
        raise ZeroDivisionError('gate_speed_factor must be non-zero.')
    d['t_gate_per_shot_s'] = 0.0165 * d['gate_speed_factor'].astype(float)

    cli_speed = d['client_gate_speed_factor'] if 'client_gate_speed_factor' in d.columns else d['gate_speed_factor']
    d['t_cli_per_shot_s'] = 0.00235 * cli_speed.astype(float)
    d['t_srv_per_shot_s'] = 0.00948 * d['gate_speed_factor'].astype(float)

    if 'shots' not in d.columns:
        d['shots'] = 1.0
    d['shots'] = d['shots'].astype(float)

    d['t_distance_total_s'] = d['t_distance_per_shot_s'] * d['shots']
    d['t_entangle_total_s'] = d['t_entangle_per_shot_s'] * d['shots']
    d['t_cli_total_s'] = d['t_cli_per_shot_s'] * d['shots']
    d['t_srv_total_s'] = d['t_srv_per_shot_s'] * d['shots']

    per_shot_cols = ['t_distance_per_shot_s', 't_entangle_per_shot_s', 't_srv_per_shot_s', 't_cli_per_shot_s']
    if 'L_total_s' not in d.columns:
        d['L_value_per_shot_s'] = d[per_shot_cols].max(axis=1)
        d['L_total_s'] = d['L_value_per_shot_s'] * d['shots']

    if 'U_total_s' not in d.columns:
        tsrv_zz = d['t_srv_per_shot_s']
        tsrv_xx = tsrv_zz + 0.000135
        d['U_value_per_shot_s'] = (
            tsrv_zz + d['t_distance_per_shot_s'] + d['t_entangle_per_shot_s'] + d['t_cli_per_shot_s']
            + tsrv_xx + d['t_distance_per_shot_s'] + d['t_entangle_per_shot_s'] + d['t_cli_per_shot_s']
        )
        d['U_total_s'] = d['U_value_per_shot_s'] * d['shots']

    return d


def select_rank_sum(chunk: pd.DataFrame) -> int:
    rank_df = pd.DataFrame(
        {
            'rank_tsrv': chunk['t_srv_total_s'].rank(method='dense', ascending=True),
            'rank_tcc': chunk['t_distance_total_s'].rank(method='dense', ascending=True),
            'rank_tent': chunk['t_entangle_total_s'].rank(method='dense', ascending=True),
            't_srv_total_s': chunk['t_srv_total_s'].to_numpy(),
            't_distance_total_s': chunk['t_distance_total_s'].to_numpy(),
            't_entangle_total_s': chunk['t_entangle_total_s'].to_numpy(),
            'order_id': np.arange(len(chunk)),
        },
        index=chunk.index,
    )

    rank_df['rank_sum'] = rank_df[['rank_tsrv', 'rank_tcc', 'rank_tent']].sum(axis=1)
    best_idx = rank_df.sort_values(
        [
            'rank_sum',
            'rank_tsrv',
            'rank_tcc',
            'rank_tent',
            't_srv_total_s',
            't_distance_total_s',
            't_entangle_total_s',
            'order_id',
        ]
    ).index[0]
    return best_idx


def summarize_regret_rate_bootstrap_ext(df: pd.DataFrame, group_sizes, n_inst=1500, n_boot=1500, alpha=0.05, seed=42) -> pd.DataFrame:
    d = _ensure_components(df)
    include_w = 'weighted_total_s' in d.columns
    selectors = ['L', 'U', 'Rank'] + (['W'] if include_w else [])
    records = []
    n_rows = len(d)

    group_sizes = sorted({int(s) for s in group_sizes if 1 <= int(s)})
    for size in group_sizes:
        if size > n_rows:
            continue

        avail_groups = n_rows // size
        n_groups = int(n_inst)
        rng = np.random.default_rng(seed + size * 1009)
        rates = {k: [] for k in selectors}

        for _ in range(n_groups):
            idx = rng.choice(n_rows, size=size, replace=False)
            chunk = d.iloc[idx]
            idx_actual = chunk['zz_time_total_s'].idxmin()
            actual_t = float(chunk.loc[idx_actual, 'zz_time_total_s'])
            if actual_t <= 0:
                continue

            idx_l = chunk['L_total_s'].idxmin()
            idx_u = chunk['U_total_s'].idxmin()
            idx_rank = select_rank_sum(chunk)
            if include_w:
                idx_w = chunk['weighted_total_s'].idxmin()

            vals = {
                'L': float(chunk.loc[idx_l, 'zz_time_total_s']),
                'U': float(chunk.loc[idx_u, 'zz_time_total_s']),
                'Rank': float(chunk.loc[idx_rank, 'zz_time_total_s']),
            }
            if include_w:
                vals['W'] = float(chunk.loc[idx_w, 'zz_time_total_s'])

            for key, value in vals.items():
                rates[key].append(abs(value - actual_t) / actual_t * 100.0)

        rec = {'group_size': size, 'total_groups': int(avail_groups), 'n_inst': int(n_groups)}
        seed_offsets = {'L': 1, 'U': 2, 'Rank': 7, 'W': 8}
        for key in selectors:
            seed_off = int(seed_offsets.get(key, 0))
            mean, lo, hi = bootstrap_ci_mean(
                rates[key],
                n_boot=n_boot,
                alpha=alpha,
                seed=seed + size * 101 + seed_off,
            )
            rec[f'{key}_mean'] = mean
            rec[f'{key}_lo'] = lo
            rec[f'{key}_hi'] = hi
        records.append(rec)

    return pd.DataFrame.from_records(records)


# =========================
# Weights + derived column
# =========================
w_raw = {
    'w_ent': 0.03876629357684344,
    'w_cc': 0.43750676409161815,
    'w_srv': 0.5237269423315385,
}
wsum = sum(w_raw.values()) or 1.0
w = {k: float(v) / float(wsum) for k, v in w_raw.items()}

_dfw = _ensure_components(df_random)
_dfw['weighted_total_s'] = (
    w['w_ent'] * _dfw['t_entangle_total_s']
    + w['w_cc'] * _dfw['t_distance_total_s']
    + w['w_srv'] * _dfw['t_srv_total_s']
)

# =========================
# Compute summaries (5-step)
# =========================
rr_ci = summarize_regret_rate_bootstrap_ext(
    _dfw,
    GROUP_SIZES,
    n_inst=N_INST,
    n_boot=N_BOOT,
    alpha=CI_ALPHA,
    seed=SEED,
)
KYOUYUUYOU_EVEN_RRCI = rr_ci.copy()

cache_name = (
    f'{CSV_RANDOM_PATH.stem}_kyouyuuyou_rrci_'
    f'g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_sd{SEED}.csv'
)
cache_path = CACHE_DIR / cache_name
rr_ci.to_csv(cache_path, index=False)

KYOUYUUYOU_EVEN_META = {
    'csv_path': str(CSV_RANDOM_PATH),
    'csv_stem': CSV_RANDOM_PATH.stem,
    'group_stride': int(GROUP_STRIDE),
    'max_group_size': int(MAX_GROUP_SIZE),
    'n_boot': int(N_BOOT),
    'n_inst': int(N_INST),
    'ci_alpha': float(CI_ALPHA),
    'seed': int(SEED),
    'weights': {k: float(v) for k, v in w.items()},
    'rr_ci_cache': str(cache_path),
    'errorbar_capsize': float(ERRORBAR_CAPSIZE),
    'errorbar_alpha': float(ERRORBAR_ALPHA),
    'ci_band_alpha': float(CI_BAND_ALPHA),
}

print(f'Saved rr_ci cache to {cache_path}')
print('Run the next cell for visualization only.')


In [ ]:
# === kyouyuuyou (even): plot only (from cache; no recompute) === fig4 vis
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# =========================
# Settings (edit here)
# =========================
MAX_GROUP_SIZE = 100
GROUP_STRIDE = 1
N_BOOT = 1500
SEED = 42
ERRORBAR_CAPSIZE = 4.0
ERRORBAR_ALPHA = 0.9
ERRORBAR_X_VALUES = [20, 40, 60, 80, 100]
CI_BAND_ALPHA = 0.18

# =========================
# Style / rcParams (keep vibe)
# =========================
plt.rcdefaults()
FONT_SCALE = 1.1
AXIS_LABEL_SCALE = 1.3 * 0.9
orig_font_size = 18
orig_axes_labelsize = 20
orig_axes_titlesize = 22
orig_legend_fontsize = 16
orig_tick_labelsize = 15

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'dejavusans',
    'font.size': orig_font_size * FONT_SCALE,
    'axes.labelsize': orig_axes_labelsize * AXIS_LABEL_SCALE,
    'axes.titlesize': orig_axes_titlesize * FONT_SCALE,
    'legend.fontsize': orig_legend_fontsize * FONT_SCALE,
    'xtick.labelsize': orig_tick_labelsize * FONT_SCALE,
    'ytick.labelsize': orig_tick_labelsize * FONT_SCALE,
})

# =========================
# Load cache (no recompute)
# =========================
CACHE_DIR = Path('data/_cache_rrci')
DEFAULT_CSV_RANDOM_PATH = Path('../../data/dr_even_base10000_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv')
default_stem = DEFAULT_CSV_RANDOM_PATH.stem

meta = globals().get('KYOUYUUYOU_EVEN_META', {})
plot_stem = meta.get('csv_stem') if isinstance(meta, dict) else None
cache_from_meta = None
if isinstance(meta, dict) and meta.get('rr_ci_cache'):
    cache_from_meta = Path(meta['rr_ci_cache'])
if isinstance(meta, dict) and 'ci_band_alpha' in meta:
    CI_BAND_ALPHA = float(meta['ci_band_alpha'])

rr_ci = None
if 'KYOUYUUYOU_EVEN_RRCI' in globals():
    in_mem = globals()['KYOUYUUYOU_EVEN_RRCI']
    if isinstance(in_mem, pd.DataFrame) and not in_mem.empty:
        rr_ci = in_mem.copy()
        print('Using rr_ci from memory: KYOUYUUYOU_EVEN_RRCI')

if rr_ci is None and cache_from_meta is not None and cache_from_meta.exists():
    rr_ci = pd.read_csv(cache_from_meta)
    print(f'Loaded rr_ci cache from meta: {cache_from_meta}')
    if not plot_stem:
        plot_stem = cache_from_meta.name.split('_kyouyuuyou_rrci_')[0]

if rr_ci is None:
    expected_cache = CACHE_DIR / (
        f'{default_stem}_kyouyuuyou_rrci_'
        f'g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_sd{SEED}.csv'
    )
    if expected_cache.exists():
        cache_path = expected_cache
    else:
        fallback = sorted(
            CACHE_DIR.glob(
                f'*_kyouyuuyou_rrci_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_sd{SEED}.csv'
            )
        )
        if not fallback:
            raise FileNotFoundError(
                f'Missing cache: {expected_cache}. Run the fig4 compute cell first.'
            )
        cache_path = fallback[-1]
    rr_ci = pd.read_csv(cache_path)
    print(f'Loaded rr_ci cache: {cache_path}')
    if not plot_stem:
        plot_stem = cache_path.name.split('_kyouyuuyou_rrci_')[0]

if not plot_stem:
    plot_stem = default_stem

required_cols = {'group_size', 'L_mean', 'L_lo', 'L_hi', 'U_mean', 'U_lo', 'U_hi', 'Rank_mean', 'Rank_lo', 'Rank_hi'}
missing = [c for c in required_cols if c not in rr_ci.columns]
if missing:
    raise KeyError(f'rr_ci cache is missing columns: {missing}')

# =========================
# Visual mapping
# =========================
colors = {
    'L': '#4D4D4D',
    'U': '#377EB8',
    'Rank': '#984EA3',
    'W': '#E41A1C',
}
label_map = {
    'L': r'$T_{min}$',
    'U': r'$T_{max}$',
    'Rank': 'Rank-sum',
    'W': 'Weighted-sum',
}
COMPOSITE_KEYS = ['U', 'W', 'Rank', 'L']
STYLE = {
    'L': dict(ls='--', mk='v'),
    'U': dict(ls='-', mk='^'),
    'Rank': dict(ls='--', mk='D'),
    'W': dict(ls='-', mk='o'),
}
ENDPOINT_MARKER = False
SHADE_ONLY_KEYS = {'U', 'W'}


def clip_nonneg(arr):
    return np.clip(np.asarray(arr, dtype=float), 0.0, None)


def plot_selector_line(ax, x, y_mean, key, label, lw=2.4):
    st = STYLE.get(key, dict(ls='-', mk='o'))
    color = colors.get(key, '#333333')
    ax.plot(
        x,
        y_mean,
        label=label,
        color=color,
        linewidth=lw,
        linestyle=st['ls'],
        marker=st['mk'] if ENDPOINT_MARKER else None,
        markevery=[-1] if ENDPOINT_MARKER else None,
        markersize=10.2,
        markeredgewidth=1.3,
        zorder=3,
    )


def draw_errorbar(ax, x, y_mean, y_lo, y_hi, key, lw=2.4):
    color = colors.get(key, '#333333')
    x_arr = np.asarray(x, dtype=float)
    y_mean_arr = np.asarray(y_mean, dtype=float)
    y_lo_arr = np.asarray(y_lo, dtype=float)
    y_hi_arr = np.asarray(y_hi, dtype=float)
    yerr = np.vstack([y_mean_arr - y_lo_arr, y_hi_arr - y_mean_arr])
    yerr = np.clip(yerr, 0.0, np.inf)

    mask = np.isin(np.rint(x_arr).astype(int), ERRORBAR_X_VALUES)
    if np.any(mask):
        ax.errorbar(
            x_arr[mask],
            y_mean_arr[mask],
            yerr=yerr[:, mask],
            fmt='none',
            ecolor=color,
            elinewidth=lw * 0.6,
            capsize=ERRORBAR_CAPSIZE,
            capthick=lw * 0.6,
            alpha=ERRORBAR_ALPHA,
            zorder=2,
        )


def draw_band(ax, x, y_lo, y_hi, key):
    color = colors.get(key, '#333333')
    y_low = np.minimum(np.asarray(y_lo, dtype=float), np.asarray(y_hi, dtype=float))
    y_high = np.maximum(np.asarray(y_lo, dtype=float), np.asarray(y_hi, dtype=float))
    ax.fill_between(x, y_low, y_high, color=color, alpha=CI_BAND_ALPHA, edgecolor='none', zorder=1)


# =========================
# Regret Rate (vertical)
# =========================
FIG_W = 11.5
FIG_H = 8.2
ymin = 0.0

TEXT_SCALE = 1.1
label_fs_v = plt.rcParams.get('axes.labelsize', 20) * 1.25 / 1.2 * 0.95 * TEXT_SCALE
label_pad_v = 2
tick_fs_v = max(
    plt.rcParams.get('xtick.labelsize', 15),
    plt.rcParams.get('ytick.labelsize', 15),
) * 1.35 * TEXT_SCALE
legend_fs_v = plt.rcParams.get('legend.fontsize', 16) * (1.10 * 1.15) * 1.015 * TEXT_SCALE

x = rr_ci['group_size'].to_numpy()
keys = [c for c in COMPOSITE_KEYS if f'{c}_mean' in rr_ci.columns]

y_cols = [c for c in rr_ci.columns if c.endswith('_mean') or c.endswith('_lo') or c.endswith('_hi')]
vals = np.concatenate([clip_nonneg(rr_ci[c]) for c in y_cols], axis=None) if len(rr_ci) and y_cols else np.array([])
ymax = float(np.nanmax(vals)) if vals.size else 1.0
if not np.isfinite(ymax) or ymax <= ymin:
    ymax = ymin + 1.0


def draw_regret_vertical(mode='errorbar', out_pdf=None):
    if mode not in {'errorbar', 'uw_band_only'}:
        raise ValueError(f"mode must be 'errorbar' or 'uw_band_only', got: {mode}")

    fig_v, ax = plt.subplots(1, 1, figsize=(FIG_W, FIG_H))

    for key in keys:
        y_mean = clip_nonneg(rr_ci[f'{key}_mean'])
        y_lo = clip_nonneg(rr_ci[f'{key}_lo'])
        y_hi = clip_nonneg(rr_ci[f'{key}_hi'])

        plot_selector_line(ax, x, y_mean, key, label_map.get(key, key), lw=2.4)

        if mode == 'errorbar':
            draw_errorbar(ax, x, y_mean, y_lo, y_hi, key, lw=2.4)
        elif key in SHADE_ONLY_KEYS:
            draw_band(ax, x, y_lo, y_hi, key)

    ax.set_xlabel(r'$N_{\mathrm{cand}}$ [count]', fontsize=label_fs_v, labelpad=label_pad_v)
    ax.set_ylabel('Regret Rate [%]', fontsize=label_fs_v, labelpad=label_pad_v)
    ax.tick_params(axis='both', which='major', labelsize=tick_fs_v, length=12, width=1.6)
    ax.tick_params(axis='both', which='minor', length=7, width=1.2)

    ax.set_ylim(ymin, ymax * 1.1)
    ax.grid(False, which='both', axis='y')
    ax.margins(x=0.02)

    handles_v, legend_labels_v = ax.get_legend_handles_labels()
    if legend_labels_v:
        leg_v = ax.legend(
            handles_v,
            legend_labels_v,
            loc='upper right',
            bbox_to_anchor=(0.998, 0.998),
            ncol=1,
            frameon=True,
            fontsize=legend_fs_v * 0.95,
            columnspacing=0.8,
            handletextpad=0.35,
            labelspacing=0.25,
            handlelength=1.5,
            borderpad=0.20,
            borderaxespad=0.15,
        )
        leg_v.get_frame().set_edgecolor('black')
        leg_v.get_frame().set_linewidth(0.4)
        leg_v.get_frame().set_alpha(1.0)

    fig_v.subplots_adjust(top=0.95, bottom=0.21)
    out_pdf = Path(out_pdf)
    fig_v.savefig(out_pdf)
    print(f'Saved ({mode}) to {out_pdf}')
    plt.show()


pdf_path_v = Path(f'{plot_stem}_regret_vertical_stride{GROUP_STRIDE}.pdf')
draw_regret_vertical(mode='errorbar', out_pdf=pdf_path_v)

pdf_path_v_uw = Path(f'{plot_stem}_regret_vertical_stride{GROUP_STRIDE}_uw_shade_only.pdf')
draw_regret_vertical(mode='uw_band_only', out_pdf=pdf_path_v_uw)


In [ ]:
# === Bottleneck-3 (cache build) === production
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Input CSVs (1000-part merged -> 10000 rows)
SCENARIOS = {
    'Tcc': {
        'csv': Path('../../data/dr_even_dist10x_10000_jit-0_cgsf-1_dist-1000-10000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv'),
        'title': 'Tcc bottleneck',
        'w_raw': {'w_ent': 4.702389786867153e-05, 'w_cc': 0.9878493820842228, 'w_srv': 0.012103594017908632},
    },
    'Tent': {
        'csv': Path('../../data/dr_even_esf0p1x_10000_jit-0_cgsf-1_dist-100-1000_esf-20-200_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv'),
        'title': 'Tent bottleneck',
        'w_raw': {'w_ent': 0.9991068260663942, 'w_cc': 0.0002547965236137298, 'w_srv': 0.0006383774099920722},
    },
    'Tsrv': {
        'csv': Path('../../data/dr_even_gsf10x_10000_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-2_63-26_3_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv'),
        'title': 'Tsrv bottleneck',
        'w_raw': {'w_ent': 4.629591708859326e-15, 'w_cc': 0.007144208142211845, 'w_srv': 0.9928557918577835},
    },
}

MAX_GROUP_SIZE = 100
GROUP_STRIDE = 1
GROUP_SIZES = [1] + list(range(GROUP_STRIDE, MAX_GROUP_SIZE + 1, GROUP_STRIDE))
GROUP_SIZES = sorted({int(s) for s in GROUP_SIZES if 1 <= int(s) <= int(MAX_GROUP_SIZE)})
N_BOOT = 1500
N_INST = 1500
ALPHA = 0.05
SEED = 42
alpha_tag = str(ALPHA).replace('.', 'p')

CACHE_DIR = Path('data/_cache_rrci_bottleneck3')
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def resolve_path(p: Path) -> Path:
    candidates = [p, Path.cwd() / p]
    s = str(p)
    if s.startswith('pb-vqe2/'):
        stripped = Path(s.replace('pb-vqe2/', '', 1))
        candidates.extend([stripped, Path.cwd() / stripped])
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(f'CSV not found: {p}')


def normalize_weights(w_raw):
    wsum = sum(w_raw.values()) or 1.0
    return {k: float(v) / float(wsum) for k, v in w_raw.items()}


def bootstrap_ci_mean(values, n_boot=1500, alpha=0.05, seed=42, chunk=256):
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    b0 = 0
    while b0 < n_boot:
        k = min(chunk, n_boot - b0)
        idx = rng.integers(0, n, size=(k, n))
        boot[b0:b0 + k] = arr[idx].mean(axis=1)
        b0 += k
    lo = np.percentile(boot, 100 * (alpha / 2.0))
    hi = np.percentile(boot, 100 * (1.0 - alpha / 2.0))
    return float(arr.mean()), float(lo), float(hi)


def _ensure_components(df):
    d = df.copy()

    if 'zz_time_total_s' not in d.columns:
        if 'zz_time_total_ns' in d.columns:
            d['zz_time_total_s'] = d['zz_time_total_ns'].astype(float) / 1e9
        elif 'zz_time' in d.columns:
            d['zz_time_total_s'] = d['zz_time'].astype(float) / 1e9
        elif 'total_time' in d.columns:
            d['zz_time_total_s'] = d['total_time'].astype(float) / 1e9
        else:
            raise KeyError('Cannot derive zz_time_total_s')

    d['t_distance_per_shot_s'] = 5.0 * d['distance'].astype(float) / 200000.0

    esf = d['entanglement_speed_factor'].astype(float)
    if (esf == 0).any():
        raise ZeroDivisionError('entanglement_speed_factor must be non-zero')
    d['t_entangle_per_shot_s'] = 5.0 / esf

    gsf = d['gate_speed_factor'].astype(float)
    if (gsf == 0).any():
        raise ZeroDivisionError('gate_speed_factor must be non-zero')
    d['t_gate_per_shot_s'] = 0.0165 * gsf

    cli_speed = d['client_gate_speed_factor'] if 'client_gate_speed_factor' in d.columns else d['gate_speed_factor']
    d['t_cli_per_shot_s'] = 0.00235 * cli_speed.astype(float)
    d['t_srv_per_shot_s'] = 0.00948 * gsf

    if 'shots' not in d.columns:
        d['shots'] = 1.0
    d['shots'] = d['shots'].astype(float)

    d['t_distance_total_s'] = d['t_distance_per_shot_s'] * d['shots']
    d['t_entangle_total_s'] = d['t_entangle_per_shot_s'] * d['shots']
    d['t_cli_total_s'] = d['t_cli_per_shot_s'] * d['shots']
    d['t_srv_total_s'] = d['t_srv_per_shot_s'] * d['shots']

    per_shot_cols = [
        't_distance_per_shot_s',
        't_entangle_per_shot_s',
        't_srv_per_shot_s',
        't_cli_per_shot_s',
    ]
    if 'L_total_s' not in d.columns:
        d['L_value_per_shot_s'] = d[per_shot_cols].max(axis=1)
        d['L_total_s'] = d['L_value_per_shot_s'] * d['shots']

    if 'U_total_s' not in d.columns:
        tsrv_zz = d['t_srv_per_shot_s']
        tsrv_xx = tsrv_zz + 0.000135
        d['U_value_per_shot_s'] = (
            tsrv_zz + d['t_distance_per_shot_s'] + d['t_entangle_per_shot_s'] + d['t_cli_per_shot_s']
            + tsrv_xx + d['t_distance_per_shot_s'] + d['t_entangle_per_shot_s'] + d['t_cli_per_shot_s']
        )
        d['U_total_s'] = d['U_value_per_shot_s'] * d['shots']

    return d


def add_weighted_total(df, w_raw):
    w = normalize_weights(w_raw)
    d = df.copy()
    d['weighted_total_s'] = (
        w['w_ent'] * d['t_entangle_total_s']
        + w['w_cc'] * d['t_distance_total_s']
        + w['w_srv'] * d['t_srv_total_s']
    )
    return d, w


def select_rank_sum(chunk):
    rank_df = pd.DataFrame(
        {
            'rank_tsrv': chunk['t_srv_total_s'].rank(method='dense', ascending=True),
            'rank_tcc': chunk['t_distance_total_s'].rank(method='dense', ascending=True),
            'rank_tent': chunk['t_entangle_total_s'].rank(method='dense', ascending=True),
            't_srv_total_s': chunk['t_srv_total_s'].to_numpy(),
            't_distance_total_s': chunk['t_distance_total_s'].to_numpy(),
            't_entangle_total_s': chunk['t_entangle_total_s'].to_numpy(),
            'order_id': np.arange(len(chunk)),
        },
        index=chunk.index,
    )
    rank_df['rank_sum'] = rank_df[['rank_tsrv', 'rank_tcc', 'rank_tent']].sum(axis=1)
    best_idx = rank_df.sort_values(
        [
            'rank_sum',
            'rank_tsrv',
            'rank_tcc',
            'rank_tent',
            't_srv_total_s',
            't_distance_total_s',
            't_entangle_total_s',
            'order_id',
        ]
    ).index[0]
    return best_idx


def summarize_regret_rate_4(df, group_sizes, n_inst=1500, n_boot=1500, alpha=0.05, seed=42):
    d = df.copy()
    selectors = ['L', 'U', 'Rank', 'W']
    records = []
    n_rows = len(d)

    group_sizes = sorted({int(s) for s in group_sizes if 1 <= int(s)})
    for size in group_sizes:
        if size > n_rows:
            continue

        avail_groups = n_rows // size
        n_groups = int(n_inst)
        rng = np.random.default_rng(seed + size * 1009)
        rates = {k: [] for k in selectors}

        for _ in range(n_groups):
            idx = rng.choice(n_rows, size=size, replace=False)
            chunk = d.iloc[idx]
            idx_actual = chunk['zz_time_total_s'].idxmin()
            actual_t = float(chunk.loc[idx_actual, 'zz_time_total_s'])
            if actual_t <= 0:
                continue

            idx_L = chunk['L_total_s'].idxmin()
            idx_U = chunk['U_total_s'].idxmin()
            idx_Rank = select_rank_sum(chunk)
            idx_W = chunk['weighted_total_s'].idxmin()

            vals = {
                'L': float(chunk.loc[idx_L, 'zz_time_total_s']),
                'U': float(chunk.loc[idx_U, 'zz_time_total_s']),
                'Rank': float(chunk.loc[idx_Rank, 'zz_time_total_s']),
                'W': float(chunk.loc[idx_W, 'zz_time_total_s']),
            }

            for k, v in vals.items():
                rates[k].append(abs(v - actual_t) / actual_t * 100.0)

        rec = {'group_size': size, 'total_groups': int(avail_groups), 'n_inst': int(n_groups)}
        seed_offsets = {'L': 1, 'U': 2, 'Rank': 7, 'W': 8}
        for k in selectors:
            seed_off = int(seed_offsets.get(k, 0))
            m, lo, hi = bootstrap_ci_mean(
                rates[k],
                n_boot=n_boot,
                alpha=alpha,
                seed=seed + size * 101 + seed_off,
            )
            rec[f'{k}_mean'] = m
            rec[f'{k}_lo'] = lo
            rec[f'{k}_hi'] = hi
        records.append(rec)

    return pd.DataFrame.from_records(records)


B3_META = {}

for key, cfg in SCENARIOS.items():
    p = resolve_path(cfg['csv'])
    print(f'[{key}] reading {p}')

    d = pd.read_csv(p)
    d = _ensure_components(d)
    d, w = add_weighted_total(d, cfg['w_raw'])

    rr_ci = summarize_regret_rate_4(d, GROUP_SIZES, n_inst=N_INST, n_boot=N_BOOT, alpha=ALPHA, seed=SEED)

    cache_name = f'b3_{key}_rrci_vec_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_a{alpha_tag}_sd{SEED}.csv'
    cache_path = CACHE_DIR / cache_name
    rr_ci.to_csv(cache_path, index=False)

    B3_META[key] = {
        'title': cfg['title'],
        'cache': str(cache_path),
        'w_norm': {k: float(v) for k, v in w.items()},
    }

    print(f'[{key}] cached -> {cache_path}')

print('B3_META ready for plot-only cell.')
for k, v in B3_META.items():
    print(k, v)



In [ ]:
# === Bottleneck-3 (plot only; no recompute) === fig5
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Re-declare minimal metadata so this cell is runnable standalone
SCENARIOS = {
    'Tcc': {
        'title': 'Tcc bottleneck',
        'w_raw': {'w_ent': 4.702389786867153e-05, 'w_cc': 0.9878493820842228, 'w_srv': 0.012103594017908632},
    },
    'Tent': {
        'title': 'Tent bottleneck',
        'w_raw': {'w_ent': 0.9991068260663942, 'w_cc': 0.0002547965236137298, 'w_srv': 0.0006383774099920722},
    },
    'Tsrv': {
        'title': 'Tsrv bottleneck',
        'w_raw': {'w_ent': 4.629591708859326e-15, 'w_cc': 0.007144208142211845, 'w_srv': 0.9928557918577835},
    },
}

MAX_GROUP_SIZE = 100
GROUP_STRIDE = 1
N_BOOT = 1500
N_INST = 1500
ALPHA = 0.05
SEED = 42
alpha_tag = str(ALPHA).replace('.', 'p')
CACHE_DIR = Path('data/_cache_rrci_bottleneck3')

PLOT_KEYS = ['U', 'W', 'Rank', 'L']
label_map = {
    'L': r'$T_{\min}$',
    'U': r'$T_{\max}$',
    'Rank': 'Rank-sum',
    'W': 'Weighted-sum',
}
colors = {
    'L': '#4D4D4D',
    'U': '#377EB8',
    'Rank': '#984EA3',
    'W': '#E41A1C',
}
styles = {
    'L': '--',
    'U': '-',
    'Rank': '--',
    'W': '-',
}

FONT_LABEL = 19 * 1.1
FONT_TICK = 20
FONT_LEGEND = 26 * 0.95
LINE_WIDTH = 2.4
CI_ALPHA = 0.15
ERRORBAR_CAPSIZE = 4.0
ERRORBAR_ALPHA = 0.9
ERRORBAR_X_VALUES = [20, 40, 60, 80, 100]
TICK_LEN_MAJOR = 14
TICK_LEN_MINOR = 8
TICK_WIDTH_MAJOR = 2.0
TICK_WIDTH_MINOR = 1.5


def clip_pos(arr, floor=1.0):
    return np.clip(np.asarray(arr, dtype=float), floor, None)


def draw_uncertainty(ax, x, y_mean, y_lo, y_hi, color, mode='errorbar', lw=LINE_WIDTH):
    x_arr = np.asarray(x, dtype=float)
    y_mean_arr = np.asarray(y_mean, dtype=float)
    y_lo_arr = np.asarray(y_lo, dtype=float)
    y_hi_arr = np.asarray(y_hi, dtype=float)

    if mode == 'errorbar':
        yerr = np.vstack([y_mean_arr - y_lo_arr, y_hi_arr - y_mean_arr])
        yerr = np.clip(yerr, 0.0, np.inf)
        mask = np.isin(np.rint(x_arr).astype(int), ERRORBAR_X_VALUES)
        if np.any(mask):
            ax.errorbar(
                x_arr[mask],
                y_mean_arr[mask],
                yerr=yerr[:, mask],
                fmt='none',
                ecolor=color,
                elinewidth=lw * 0.6,
                capsize=ERRORBAR_CAPSIZE,
                capthick=lw * 0.6,
                alpha=ERRORBAR_ALPHA,
                zorder=2,
            )
    elif mode == 'band':
        y_low = np.minimum(y_lo_arr, y_hi_arr)
        y_high = np.maximum(y_lo_arr, y_hi_arr)
        ax.fill_between(x_arr, y_low, y_high, color=color, alpha=CI_ALPHA, edgecolor='none', zorder=1)
    else:
        raise ValueError(f"mode must be 'errorbar' or 'band', got: {mode}")


scenario_order = ['Tcc', 'Tent', 'Tsrv']
cache_map = {
    k: CACHE_DIR / f'b3_{k}_rrci_vec_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_a{alpha_tag}_sd{SEED}.csv' for k in scenario_order
}

rrci = {}
for k in scenario_order:
    cp = cache_map[k]
    if not cp.exists():
        fallback = sorted(CACHE_DIR.glob(f'b3_{k}_rrci_vec_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b*_a{alpha_tag}_sd{SEED}.csv'))
        if fallback:
            cp = fallback[-1]
        else:
            raise FileNotFoundError(
                f'Missing cache: {cp}. Run the cache-build cell first.'
            )
    rrci[k] = pd.read_csv(cp)

all_vals = []
for k in scenario_order:
    df = rrci[k]
    for pk in PLOT_KEYS:
        for suf in ['_mean', '_lo', '_hi']:
            col = f'{pk}{suf}'
            if col in df.columns:
                all_vals.append(clip_pos(df[col]))

ymax_global = float(np.concatenate(all_vals).max()) if all_vals else 1.0


def draw_bottleneck3(mode: str, out_pdf: str):
    fig, axes = plt.subplots(1, 3, figsize=(19.6, 7.92), sharex=True, sharey=True)

    for i, key in enumerate(scenario_order):
        ax = axes[i]
        data = rrci[key]
        x = data['group_size'].to_numpy()

        for pk in PLOT_KEYS:
            y_mean = clip_pos(data[f'{pk}_mean'])
            y_lo = clip_pos(data[f'{pk}_lo'])
            y_hi = clip_pos(data[f'{pk}_hi'])
            label = label_map[pk] if i == 0 else None
            ax.plot(x, y_mean, color=colors[pk], linestyle=styles[pk], linewidth=LINE_WIDTH, label=label, zorder=3)
            draw_uncertainty(ax, x, y_mean, y_lo, y_hi, color=colors[pk], mode=mode, lw=LINE_WIDTH)

        ax.set_yscale('linear')
        ax.set_ylim(1.0, min(ymax_global * 1.2, 80.0))
        ax.grid(False, which='both', axis='y')
        ax.margins(x=0.04)
        ax.tick_params(
            axis='both',
            which='major',
            labelsize=FONT_TICK,
            pad=6,
            length=TICK_LEN_MAJOR,
            width=TICK_WIDTH_MAJOR,
        )
        ax.tick_params(
            axis='both',
            which='minor',
            length=TICK_LEN_MINOR,
            width=TICK_WIDTH_MINOR,
        )
        ax.set_xlabel('# of server candidates', fontsize=FONT_LABEL, labelpad=4)

    axes[0].set_ylabel('Regret Rate [%]', fontsize=FONT_LABEL, labelpad=6)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc='upper center',
        ncol=len(labels),
        frameon=False,
        bbox_to_anchor=(0.5, 0.972),
        fontsize=FONT_LEGEND,
        handlelength=3.2,
        handletextpad=0.5,
        columnspacing=1.4,
    )

    fig.subplots_adjust(top=0.87, bottom=0.20, left=0.09, right=0.995, wspace=0.12)
    out_pdf = Path(out_pdf)
    fig.savefig(out_pdf, bbox_inches='tight', pad_inches=0.08)
    print(f"Saved ({mode}) -> {out_pdf.resolve()}")
    plt.show()


# errorbar only
draw_bottleneck3(mode='errorbar', out_pdf='bottleneck3_regret_LURW_grid_1x3_errorbar.pdf')

# CI band only
draw_bottleneck3(mode='band', out_pdf='bottleneck3_regret_LURW_grid_1x3_ci_band.pdf')



In [ ]:
# === Bottleneck-3 split export (fixed-size single panels; errorbar + no-uncertainty + U/W-shade) === fig5 split
# Run this cell after the fig5 cell above.
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

required_vars = [
    'rrci', 'scenario_order', 'SCENARIOS', 'PLOT_KEYS', 'label_map',
    'colors', 'styles', 'clip_pos', 'draw_uncertainty', 'LINE_WIDTH',
    'FONT_LABEL', 'FONT_TICK', 'FONT_LEGEND', 'TICK_LEN_MAJOR',
    'TICK_LEN_MINOR', 'TICK_WIDTH_MAJOR', 'TICK_WIDTH_MINOR',
    'ymax_global',
]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        'Run the fig5 plotting cell first. Missing variables: ' + ', '.join(missing)
    )

# fig3-style fixed layout to keep all exported panels the same size.
FIGSIZE_SINGLE = (12.5, 9.8)
AXES_BOX = [0.18, 0.15, 0.78, 0.79]
AXIS_TEXT_SCALE = 1.5
LEGEND_TEXT_SCALE = 1.3
EXTRA_TEXT_SCALE = 1.13
SPLIT_SCALE_UP = 1.1
FIG5_SPLIT_TEXT_BOOST = 1.3  # axis-label + legend text only
LABEL_FONT_SPLIT = FONT_LABEL * AXIS_TEXT_SCALE * EXTRA_TEXT_SCALE * SPLIT_SCALE_UP * FIG5_SPLIT_TEXT_BOOST
TICK_FONT_SPLIT = FONT_TICK * AXIS_TEXT_SCALE * EXTRA_TEXT_SCALE
LEGEND_FONT_SPLIT = FONT_LEGEND * LEGEND_TEXT_SCALE * EXTRA_TEXT_SCALE * FIG5_SPLIT_TEXT_BOOST
LEGEND_HANDLE_LENGTH_SPLIT = 3.2 * (2.0 / 3.0) * 0.75
TICK_LABEL_TEXT_SCALE = 0.9
TICK_MARK_SCALE = 1.5
TICK_FONT_SPLIT = TICK_FONT_SPLIT * TICK_LABEL_TEXT_SCALE * SPLIT_SCALE_UP
TICK_LEN_MAJOR_SPLIT = TICK_LEN_MAJOR * TICK_MARK_SCALE * SPLIT_SCALE_UP
TICK_LEN_MINOR_SPLIT = TICK_LEN_MINOR * TICK_MARK_SCALE * SPLIT_SCALE_UP

short_name = {
    'Tcc': 'fig5_tcc',
    'Tent': 'fig5_tent',
    'Tsrv': 'fig5_tsrv',
}
SHADE_ONLY_KEYS = {'U', 'W'}

x_min = min(float(rrci[k]['group_size'].min()) for k in scenario_order)
x_max = max(float(rrci[k]['group_size'].max()) for k in scenario_order)
y_max = min(ymax_global * 1.2, 80.0)
Y_MAX_NO_RANK = 20.0  # ~20% + alpha


def draw_bottleneck3_split(uncertainty='errorbar', out_suffix='', include_rank=True, shade_keys=None, y_max_override=None):
    if uncertainty not in {'errorbar', 'none', 'uw_band_only'}:
        raise ValueError(f"uncertainty must be 'errorbar', 'none', or 'uw_band_only', got: {uncertainty}")

    if shade_keys is None:
        shade_keys = SHADE_ONLY_KEYS

    active_plot_keys = [pk for pk in PLOT_KEYS if include_rank or pk != 'Rank']
    y_max_local = float(y_max_override) if y_max_override is not None else y_max

    for key in scenario_order:
        fig = plt.figure(figsize=FIGSIZE_SINGLE)
        ax = fig.add_axes(AXES_BOX)

        data = rrci[key]
        x = data['group_size'].to_numpy()

        for pk in active_plot_keys:
            y_mean = clip_pos(data[f'{pk}_mean'])
            y_lo = clip_pos(data[f'{pk}_lo'])
            y_hi = clip_pos(data[f'{pk}_hi'])
            label = label_map[pk] if key == 'Tent' else None
            ax.plot(
                x,
                y_mean,
                color=colors[pk],
                linestyle=styles[pk],
                linewidth=LINE_WIDTH,
                label=label,
                zorder=3,
            )
            if uncertainty == 'errorbar':
                draw_uncertainty(ax, x, y_mean, y_lo, y_hi, color=colors[pk], mode='errorbar', lw=LINE_WIDTH)
            elif uncertainty == 'uw_band_only' and pk in shade_keys:
                draw_uncertainty(ax, x, y_mean, y_lo, y_hi, color=colors[pk], mode='band', lw=LINE_WIDTH)

        ax.set_xlabel('# of server candidates', fontsize=LABEL_FONT_SPLIT, labelpad=4)
        ax.set_ylabel('Regret Rate [%]', fontsize=LABEL_FONT_SPLIT, labelpad=6)
        ax.set_yscale('linear')
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(1.0, y_max_local)
        ax.grid(False, which='both', axis='y')
        ax.margins(x=0.02)
        ax.tick_params(
            axis='both',
            which='major',
            labelsize=45,
            pad=6,
            length=TICK_LEN_MAJOR_SPLIT,
            width=TICK_WIDTH_MAJOR,
        )
        ax.tick_params(
            axis='both',
            which='minor',
            length=TICK_LEN_MINOR_SPLIT,
            width=TICK_WIDTH_MINOR,
        )

        if key == 'Tent':
            leg = ax.legend(
                loc='upper right',
                bbox_to_anchor=(0.975, 0.975),
                borderaxespad=0.35,
                ncol=1,
                fontsize=LEGEND_FONT_SPLIT,
                frameon=True,
                handlelength=LEGEND_HANDLE_LENGTH_SPLIT,
                handletextpad=0.65,
                labelspacing=0.35,
                borderpad=0.45,
                columnspacing=1.1,
            )
            leg.get_frame().set_edgecolor('black')
            leg.get_frame().set_linewidth(0.4)
            leg.get_frame().set_alpha(1.0)

        out_pdf = Path(f"{short_name[key]}{out_suffix}.pdf")
        fig.savefig(out_pdf)
        print(f'Saved ({key}, {uncertainty}, include_rank={include_rank}) -> {out_pdf.resolve()}')
        plt.show()


# existing variants
draw_bottleneck3_split(uncertainty='errorbar', out_suffix='')
draw_bottleneck3_split(uncertainty='none', out_suffix='_no_uncertainty')
draw_bottleneck3_split(uncertainty='uw_band_only', out_suffix='_uw_shade_only')

# new variant: Tmax(U) + Weighted-sum(W) shaded, Rank excluded
draw_bottleneck3_split(
    uncertainty='uw_band_only',
    out_suffix='_uw_shade_only_no_rank',
    include_rank=False,
    shade_keys={'U', 'W'},
    y_max_override=Y_MAX_NO_RANK,
)


In [ ]:
# === Classic big/small 2-panel (cache build) ===　fig6
from pathlib import Path

import numpy as np
import pandas as pd

CSV = {
    "j2p9m": {
        1: Path("../../data/dr_j2p9m_jit-2_9e+06_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-1_smp-10000_set-1_sd-42_str-997_psd-42.csv"),
        10: Path("../../data/dr_j2p9m_10_jit-2_9e+06_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-42.csv"),
    },
    "j72k": {
        1: Path("../../data/dr_j72k_jit-72000_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-1_smp-10000_set-1_sd-42_str-997_psd-42.csv"),
        10: Path("../../data/dr_j72k_10_jit-72000_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-42.csv"),
    },
}

SCENARIO_LABEL = {
    "j2p9m": "Classical-large",
    "j72k": "Classical-small",
}
SCENARIO_ORDER = ["j2p9m", "j72k"]
SHOT_ORDER = [10, 1]

MAX_GROUP_SIZE = 100
GROUP_STRIDE = 1
GROUP_SIZES = [1] + list(range(GROUP_STRIDE, MAX_GROUP_SIZE + 1, GROUP_STRIDE))
GROUP_SIZES = sorted({int(s) for s in GROUP_SIZES if 1 <= int(s) <= int(MAX_GROUP_SIZE)})
N_BOOT = 1500
N_INST = 1500
ALPHA = 0.05
SEED = 42
alpha_tag = str(ALPHA).replace('.', 'p')

WEIGHT_RAW_BY_SHOTS = {
    10: {"w_ent": 0.03876629357684344, "w_cc": 0.43750676409161815, "w_srv": 0.5237269423315385},
    1: {"w_ent": 0.05503439277181065, "w_cc": 0.43988882263334483, "w_srv": 0.5050767845948445},
}
WEIGHT_RAW = WEIGHT_RAW_BY_SHOTS[10]
SELECTORS_TO_EXPORT = ["L", "U", "Rank", "W"]

RRCI_CACHE_DIR = Path("data/_cache_rrci")
RRCI_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_CACHE_DIR = Path("data/_cache_rrci_classic2panel")
OUT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def resolve_path(p: Path) -> Path:
    cands = [p, Path.cwd() / p]
    s = str(p)
    if s.startswith("pb-vqe2/"):
        stripped = Path(s.replace("pb-vqe2/", "", 1))
        cands.extend([stripped, Path.cwd() / stripped])
    for q in cands:
        if q.exists():
            return q
    raise FileNotFoundError(f"CSV not found: {p}")


def bootstrap_ci_mean(values, n_boot=1500, alpha=0.05, seed=42, chunk=256):
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    if n == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    b0 = 0
    while b0 < n_boot:
        k = min(chunk, n_boot - b0)
        idx = rng.integers(0, n, size=(k, n))
        boot[b0:b0 + k] = arr[idx].mean(axis=1)
        b0 += k

    lo = np.percentile(boot, 100 * (alpha / 2.0))
    hi = np.percentile(boot, 100 * (1.0 - alpha / 2.0))
    return float(arr.mean()), float(lo), float(hi)


def ensure_components(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    if "zz_time_total_s" not in d.columns:
        if "zz_time_total_ns" in d.columns:
            d["zz_time_total_s"] = d["zz_time_total_ns"].astype(float) / 1e9
        elif "zz_time" in d.columns:
            d["zz_time_total_s"] = d["zz_time"].astype(float) / 1e9
        elif "total_time" in d.columns:
            d["zz_time_total_s"] = d["total_time"].astype(float) / 1e9
        else:
            raise KeyError("Cannot derive zz_time_total_s")

    d["t_distance_per_shot_s"] = 5.0 * d["distance"].astype(float) / 200000.0

    esf = d["entanglement_speed_factor"].astype(float)
    if (esf == 0).any():
        raise ZeroDivisionError("entanglement_speed_factor must be non-zero")
    d["t_entangle_per_shot_s"] = 5.0 / esf

    gsf = d["gate_speed_factor"].astype(float)
    if (gsf == 0).any():
        raise ZeroDivisionError("gate_speed_factor must be non-zero")
    d["t_gate_per_shot_s"] = 0.0165 * gsf

    cli_speed = d["client_gate_speed_factor"] if "client_gate_speed_factor" in d.columns else d["gate_speed_factor"]
    d["t_cli_per_shot_s"] = 0.00235 * cli_speed.astype(float)
    d["t_srv_per_shot_s"] = 0.00948 * gsf

    if "shots" not in d.columns:
        d["shots"] = 1.0
    d["shots"] = d["shots"].astype(float)

    d["t_distance_total_s"] = d["t_distance_per_shot_s"] * d["shots"]
    d["t_entangle_total_s"] = d["t_entangle_per_shot_s"] * d["shots"]
    d["t_cli_total_s"] = d["t_cli_per_shot_s"] * d["shots"]
    d["t_srv_total_s"] = d["t_srv_per_shot_s"] * d["shots"]

    per_shot_cols = [
        "t_distance_per_shot_s",
        "t_entangle_per_shot_s",
        "t_srv_per_shot_s",
        "t_cli_per_shot_s",
    ]
    if "L_total_s" not in d.columns:
        d["L_value_per_shot_s"] = d[per_shot_cols].max(axis=1)
        d["L_total_s"] = d["L_value_per_shot_s"] * d["shots"]

    if "U_total_s" not in d.columns:
        tsrv_zz = d["t_srv_per_shot_s"]
        tsrv_xx = tsrv_zz + 0.000135
        d["U_value_per_shot_s"] = (
            tsrv_zz + d["t_distance_per_shot_s"] + d["t_entangle_per_shot_s"] + d["t_cli_per_shot_s"]
            + tsrv_xx + d["t_distance_per_shot_s"] + d["t_entangle_per_shot_s"] + d["t_cli_per_shot_s"]
        )
        d["U_total_s"] = d["U_value_per_shot_s"] * d["shots"]

    return d


def select_rank_sum(chunk: pd.DataFrame):
    rank_df = pd.DataFrame(
        {
            "rank_tsrv": chunk["t_srv_total_s"].rank(method="dense", ascending=True),
            "rank_tcc": chunk["t_distance_total_s"].rank(method="dense", ascending=True),
            "rank_tent": chunk["t_entangle_total_s"].rank(method="dense", ascending=True),
            "t_srv_total_s": chunk["t_srv_total_s"].to_numpy(),
            "t_distance_total_s": chunk["t_distance_total_s"].to_numpy(),
            "t_entangle_total_s": chunk["t_entangle_total_s"].to_numpy(),
            "order_id": np.arange(len(chunk)),
        },
        index=chunk.index,
    )
    rank_df["rank_sum"] = rank_df[["rank_tsrv", "rank_tcc", "rank_tent"]].sum(axis=1)
    best_idx = rank_df.sort_values(
        [
            "rank_sum",
            "rank_tsrv",
            "rank_tcc",
            "rank_tent",
            "t_srv_total_s",
            "t_distance_total_s",
            "t_entangle_total_s",
            "order_id",
        ]
    ).index[0]
    return best_idx


def summarize_regret_rate_bootstrap(df: pd.DataFrame, group_sizes, n_inst=1500, n_boot=1500, alpha=0.05, seed=42) -> pd.DataFrame:
    d = ensure_components(df)

    wsum = sum(WEIGHT_RAW.values()) or 1.0
    w = {k: float(v) / float(wsum) for k, v in WEIGHT_RAW.items()}
    d["weighted_total_s"] = (
        w["w_ent"] * d["t_entangle_total_s"]
        + w["w_cc"] * d["t_distance_total_s"]
        + w["w_srv"] * d["t_srv_total_s"]
    )

    selectors = ["L", "U", "Rank", "W"]
    records = []
    n_rows = len(d)

    group_sizes = sorted({int(s) for s in group_sizes if 1 <= int(s)})
    for size in group_sizes:
        if size > n_rows:
            continue

        avail_groups = n_rows // size
        n_groups = int(n_inst)
        rng = np.random.default_rng(seed + size * 1009)
        rates = {k: [] for k in selectors}

        for _ in range(n_groups):
            idx = rng.choice(n_rows, size=size, replace=False)
            chunk = d.iloc[idx]
            idx_actual = chunk["zz_time_total_s"].idxmin()
            actual_t = float(chunk.loc[idx_actual, "zz_time_total_s"])
            if actual_t <= 0:
                continue

            idx_L = chunk["L_total_s"].idxmin()
            idx_U = chunk["U_total_s"].idxmin()
            idx_Rank = select_rank_sum(chunk)
            idx_W = chunk["weighted_total_s"].idxmin()

            vals = {
                "L": float(chunk.loc[idx_L, "zz_time_total_s"]),
                "U": float(chunk.loc[idx_U, "zz_time_total_s"]),
                "Rank": float(chunk.loc[idx_Rank, "zz_time_total_s"]),
                "W": float(chunk.loc[idx_W, "zz_time_total_s"]),
            }

            for k, v in vals.items():
                rates[k].append(abs(v - actual_t) / actual_t * 100.0)

        rec = {"group_size": size, "total_groups": int(avail_groups), "n_inst": int(n_groups)}
        seed_offsets = {"L": 1, "U": 2, "Rank": 7, "W": 8}
        for k in selectors:
            seed_off = int(seed_offsets.get(k, 0))
            m, lo, hi = bootstrap_ci_mean(
                rates[k],
                n_boot=n_boot,
                alpha=alpha,
                seed=seed + size * 101 + seed_off,
            )
            rec[f"{k}_mean"] = m
            rec[f"{k}_lo"] = lo
            rec[f"{k}_hi"] = hi
        records.append(rec)

    return pd.DataFrame.from_records(records)


def get_rrci(csv_path: Path) -> pd.DataFrame:
    p = resolve_path(csv_path)
    cache_name = f"{p.stem}_vec_rrci_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_a{alpha_tag}_sd{SEED}.csv"
    cache_path = RRCI_CACHE_DIR / cache_name

    if cache_path.exists():
        print(f"cache hit: {cache_path}")
        return pd.read_csv(cache_path)

    print(f"computing rrci: {p}")
    raw_df = pd.read_csv(p)
    rr = summarize_regret_rate_bootstrap(raw_df, GROUP_SIZES, n_inst=N_INST, n_boot=N_BOOT, alpha=ALPHA, seed=SEED)
    rr.to_csv(cache_path, index=False)
    print(f"cached rrci -> {cache_path}")
    return rr


records = []
for scen in SCENARIO_ORDER:
    for shots in SHOT_ORDER:
        WEIGHT_RAW = WEIGHT_RAW_BY_SHOTS[int(shots)]
        rr = get_rrci(CSV[scen][shots])
        for sel in SELECTORS_TO_EXPORT:
            m_col = f"{sel}_mean"
            lo_col = f"{sel}_lo"
            hi_col = f"{sel}_hi"
            if m_col not in rr.columns:
                continue
            part = rr[["group_size", m_col, lo_col, hi_col]].copy()
            part.columns = ["group_size", "mean", "lo", "hi"]
            part["scenario"] = scen
            part["scenario_label"] = SCENARIO_LABEL[scen]
            part["shots"] = int(shots)
            part["selector"] = sel
            records.append(part)

if not records:
    raise RuntimeError("No records were created. Check source rrci columns.")

plot_df = pd.concat(records, ignore_index=True)
out_csv = OUT_CACHE_DIR / f"classic_bigsmall_rrci_long_vec_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_a{alpha_tag}_sd{SEED}.csv"
plot_df.to_csv(out_csv, index=False)

CLASSIC2P_META = {
    "cache_csv": str(out_csv),
    "scenarios": SCENARIO_ORDER,
    "shots": SHOT_ORDER,
    "selectors": SELECTORS_TO_EXPORT,
    "weight_raw_by_shots": WEIGHT_RAW_BY_SHOTS,
    "group_stride": int(GROUP_STRIDE),
}

print(f"Saved plot-cache -> {out_csv.resolve()}")
print("CLASSIC2P_META:", CLASSIC2P_META)



In [ ]:
# === Classic high/low 2-panel (plot only; no recompute) ===　fig6
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MAX_GROUP_SIZE = 100
GROUP_STRIDE = 1
N_BOOT = 1500
N_INST = 1500
ALPHA = 0.05
SEED = 42
alpha_tag = str(ALPHA).replace('.', 'p')
CACHE_CSV = Path(f"data/_cache_rrci_classic2panel/classic_bigsmall_rrci_long_vec_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b{N_BOOT}_a{alpha_tag}_sd{SEED}.csv")

if not CACHE_CSV.exists():
    fallback = sorted(Path('data/_cache_rrci_classic2panel').glob(f"classic_bigsmall_rrci_long_vec_i{N_INST}_g{MAX_GROUP_SIZE}_s{GROUP_STRIDE}_b*_a{alpha_tag}_sd{SEED}.csv"))
    if fallback:
        CACHE_CSV = fallback[-1]
    else:
        raise FileNotFoundError(f"Missing cache: {CACHE_CSV}. Run the cache-build cell first.")

plot_df = pd.read_csv(CACHE_CSV)

SHOT_ORDER = [10, 1]
SCENARIO_ORDER = ["j2p9m", "j72k"]
SCENARIO_STYLE = {
    "j2p9m": {"ls": "-", "legend": "high jitter"},
    "j72k": {"ls": "--", "legend": "low jitter"},
}
SHOT_STYLE = {
    10: {"ls": "-", "legend": "shots=10"},
    1: {"ls": "--", "legend": "shots=1"},
}

SELECTOR_META = {
    "U": {"mk": "^", "name": r"$T_{max}$", "color": "#377EB8"},
    "W": {"mk": "o", "name": "Weighted", "color": "#E41A1C"},
}
SCENARIO_COLOR_OVERRIDE = {
    "j72k": {
        "U": "#FF7F00",  # orange for dashed(low)
        "W": "#4DAF4A",  # green for dashed(low)
    }
}
SHOT_SELECTOR_COLOR = {
    10: {"U": "#377EB8", "W": "#E41A1C"},
    1: {"U": "#FF7F00", "W": "#4DAF4A"},
}

FLOOR_MIN = 1.0
CI_ALPHA = 0.15
ERRORBAR_CAPSIZE = 4.0
ERRORBAR_ALPHA = 0.9
ERRORBAR_X_VALUES = [20, 40, 60, 80, 100]
LINE_WIDTH = 2.4
ENDPOINT_MARKER = False
YMAX_FIG6_HIGH = 50.0  # high-jitter panel fixed at 50%

FONT_LABEL = 24 * 1.07
FONT_TICK = 24
FONT_LEGEND = 26.4

TICK_LEN_MAJOR = 12
TICK_LEN_MINOR = 7
TICK_WIDTH_MAJOR = 2.0
TICK_WIDTH_MINOR = 1.5


def clip_pos(arr):
    return np.clip(np.asarray(arr, dtype=float), FLOOR_MIN, None)


def compute_ymax(df, selectors):
    vals = []
    sub = df[df["selector"].isin(selectors)]
    for col in ["mean", "lo", "hi"]:
        if col in sub.columns:
            vals.append(clip_pos(sub[col].to_numpy()))
    if not vals:
        return FLOOR_MIN
    return float(np.concatenate(vals).max())


def draw_errorbar_classic(ax, x, y_mean, y_lo, y_hi, ecolor, lw=LINE_WIDTH):
    x_arr = np.asarray(x, dtype=float)
    y_mean_arr = np.asarray(y_mean, dtype=float)
    y_lo_arr = np.asarray(y_lo, dtype=float)
    y_hi_arr = np.asarray(y_hi, dtype=float)

    yerr = np.vstack([y_mean_arr - y_lo_arr, y_hi_arr - y_mean_arr])
    yerr = np.clip(yerr, 0.0, np.inf)

    mask = np.isin(np.rint(x_arr).astype(int), ERRORBAR_X_VALUES)
    if np.any(mask):
        ax.errorbar(
            x_arr[mask],
            y_mean_arr[mask],
            yerr=yerr[:, mask],
            fmt="none",
            ecolor=ecolor,
            elinewidth=lw * 0.6,
            capsize=ERRORBAR_CAPSIZE,
            capthick=lw * 0.6,
            alpha=ERRORBAR_ALPHA,
            zorder=2,
        )


def scenario_color(sel, scen):
    base = SELECTOR_META[sel]["color"]
    return SCENARIO_COLOR_OVERRIDE.get(scen, {}).get(sel, base)


def plot_family(
    ax,
    data,
    selectors,
    panel_idx,
    uncertainty="errorbar",
    line_width=LINE_WIDTH,
    marker_size=10.0,
    marker_edge_width=1.3,
):
    if uncertainty not in {"errorbar", "band"}:
        raise ValueError(f"uncertainty must be 'errorbar' or 'band', got: {uncertainty}")

    for scen in SCENARIO_ORDER:
        for sel in selectors:
            d = data[(data["scenario"] == scen) & (data["selector"] == sel)].sort_values("group_size")
            if d.empty:
                continue

            x = d["group_size"].to_numpy()
            y = clip_pos(d["mean"].to_numpy())
            lo = clip_pos(d["lo"].to_numpy())
            hi = clip_pos(d["hi"].to_numpy())

            meta = SELECTOR_META[sel]
            sc = SCENARIO_STYLE[scen]
            line_color = scenario_color(sel, scen)
            label = f"{meta['name']} ({sc['legend']})" if panel_idx == 0 else None

            ax.plot(
                x,
                y,
                color=line_color,
                linestyle=sc["ls"],
                linewidth=line_width,
                marker=meta["mk"] if ENDPOINT_MARKER else None,
                markevery=[-1] if ENDPOINT_MARKER else None,
                markersize=marker_size,
                markeredgewidth=marker_edge_width,
                label=label,
                zorder=3,
            )

            if uncertainty == "errorbar":
                draw_errorbar_classic(ax, x, y, lo, hi, line_color, lw=line_width)
            else:
                band_alpha = CI_ALPHA
                y_low = np.minimum(lo, hi)
                y_high = np.maximum(lo, hi)
                ax.fill_between(x, y_low, y_high, color=line_color, alpha=band_alpha, edgecolor="none", zorder=1)


def draw_two_panel(selectors, out_pdf, yscale="linear", uncertainty="errorbar"):
    ymax = compute_ymax(plot_df, selectors)
    if yscale not in {"log", "linear"}:
        raise ValueError(f"yscale must be 'log' or 'linear', got: {yscale}")

    fig, axes = plt.subplots(1, 2, figsize=(20.8, 8.0), sharex=True, sharey=True)

    for i, shots in enumerate(SHOT_ORDER):
        ax = axes[i]
        data = plot_df[(plot_df["shots"] == shots) & (plot_df["selector"].isin(selectors))]
        plot_family(ax, data, selectors, panel_idx=i, uncertainty=uncertainty)

        ax.set_yscale(yscale)
        ax.set_ylim(FLOOR_MIN, ymax * 1.2)
        ax.grid(False, which="both", axis="y")
        ax.margins(x=0.04)

        ax.tick_params(
            axis="both",
            which="major",
            labelsize=FONT_TICK,
            pad=6,
            length=TICK_LEN_MAJOR,
            width=TICK_WIDTH_MAJOR,
        )
        ax.tick_params(
            axis="both",
            which="minor",
            length=TICK_LEN_MINOR,
            width=TICK_WIDTH_MINOR,
        )
        ax.set_xlabel("# of server candidates", fontsize=FONT_LABEL, labelpad=4)

    axes[0].set_ylabel("Regret Rate [%]", fontsize=FONT_LABEL, labelpad=6)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 0.992),
        borderaxespad=0.22,
        fontsize=FONT_LEGEND,
        handlelength=3.2,
        handletextpad=0.62,
        columnspacing=1.35,
    )

    fig.subplots_adjust(top=0.84, bottom=0.20, left=0.09, right=0.985, wspace=0.12)

    out_pdf = Path(out_pdf)
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.08)
    print(f"Saved ({uncertainty}) -> {out_pdf.resolve()}")
    plt.show()


def draw_two_panel_vertical(selectors, out_pdf, yscale="linear", uncertainty="errorbar"):
    scale = 1.3
    if yscale not in {"log", "linear"}:
        raise ValueError(f"yscale must be 'log' or 'linear', got: {yscale}")

    fig, axes = plt.subplots(2, 1, figsize=(14.0, 16.8), sharex=True, sharey=False)

    for i, shots in enumerate(SHOT_ORDER):
        ax = axes[i]
        data = plot_df[(plot_df["shots"] == shots) & (plot_df["selector"].isin(selectors))]
        plot_family(
            ax,
            data,
            selectors,
            panel_idx=i,
            uncertainty=uncertainty,
            line_width=LINE_WIDTH * scale,
            marker_size=10.0 * scale,
            marker_edge_width=1.3 * scale,
        )

        local_ymax = compute_ymax(data, selectors)
        ax.set_yscale(yscale)
        ax.set_ylim(FLOOR_MIN, local_ymax * 1.2)
        ax.grid(False, which="both", axis="y")
        ax.margins(x=0.04)

        ax.tick_params(
            axis="both",
            which="major",
            labelsize=FONT_TICK * scale,
            pad=6 * scale,
            length=TICK_LEN_MAJOR * scale,
            width=TICK_WIDTH_MAJOR * scale,
        )
        ax.tick_params(
            axis="both",
            which="minor",
            length=TICK_LEN_MINOR * scale,
            width=TICK_WIDTH_MINOR * scale,
        )
        ax.set_xlabel("# of server candidates", fontsize=FONT_LABEL * scale, labelpad=4 * scale)
        ax.set_ylabel("Regret Rate [%]", fontsize=FONT_LABEL * scale, labelpad=6 * scale)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.53, 0.980),
        borderaxespad=0.22,
        fontsize=FONT_LEGEND * scale * 0.9,
        handlelength=3.2,
        handletextpad=0.62,
        columnspacing=1.35,
    )

    fig.subplots_adjust(top=0.84, bottom=0.09, left=0.13, right=0.985, hspace=0.18)

    out_pdf = Path(out_pdf)
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.08)
    print(f"Saved ({uncertainty}) -> {out_pdf.resolve()}")
    plt.show()


def plot_family_by_shots(
    ax,
    data,
    selectors,
    panel_idx,
    uncertainty="errorbar",
    line_width=LINE_WIDTH,
    marker_size=10.0,
    marker_edge_width=1.3,
):
    if uncertainty not in {"errorbar", "band"}:
        raise ValueError(f"uncertainty must be 'errorbar' or 'band', got: {uncertainty}")

    for shots in SHOT_ORDER:
        for sel in selectors:
            d = data[(data["shots"] == int(shots)) & (data["selector"] == sel)].sort_values("group_size")
            if d.empty:
                continue

            x = d["group_size"].to_numpy()
            y = clip_pos(d["mean"].to_numpy())
            lo = clip_pos(d["lo"].to_numpy())
            hi = clip_pos(d["hi"].to_numpy())

            meta = SELECTOR_META[sel]
            sh = SHOT_STYLE[int(shots)]
            line_color = SHOT_SELECTOR_COLOR.get(int(shots), {}).get(sel, meta["color"])
            label = f"{meta['name']} ({sh['legend']})" if panel_idx == 0 else None

            ax.plot(
                x,
                y,
                color=line_color,
                linestyle=sh["ls"],
                linewidth=line_width,
                marker=meta["mk"] if ENDPOINT_MARKER else None,
                markevery=[-1] if ENDPOINT_MARKER else None,
                markersize=marker_size,
                markeredgewidth=marker_edge_width,
                label=label,
                zorder=3,
            )

            if uncertainty == "errorbar":
                draw_errorbar_classic(ax, x, y, lo, hi, line_color, lw=line_width)
            else:
                y_low = np.minimum(lo, hi)
                y_high = np.maximum(lo, hi)
                ax.fill_between(x, y_low, y_high, color=line_color, alpha=CI_ALPHA, edgecolor="none", zorder=1)


def draw_two_panel_vertical_by_scenario(selectors, out_pdf, yscale="linear", uncertainty="errorbar"):
    scale = 1.3
    if yscale not in {"log", "linear"}:
        raise ValueError(f"yscale must be 'log' or 'linear', got: {yscale}")

    fig, axes = plt.subplots(2, 1, figsize=(14.0, 16.8), sharex=True, sharey=False)

    for i, scen in enumerate(SCENARIO_ORDER):
        ax = axes[i]
        data = plot_df[(plot_df["scenario"] == scen) & (plot_df["selector"].isin(selectors))]
        plot_family_by_shots(
            ax,
            data,
            selectors,
            panel_idx=i,
            uncertainty=uncertainty,
            line_width=LINE_WIDTH * scale,
            marker_size=10.0 * scale,
            marker_edge_width=1.3 * scale,
        )

        local_ymax = compute_ymax(data, selectors)
        ax.set_yscale(yscale)
        if scen == "j2p9m":
            y_top = YMAX_FIG6_HIGH
        else:
            y_top = local_ymax * 1.2
        ax.set_ylim(FLOOR_MIN, y_top)
        ax.grid(False, which="both", axis="y")
        ax.margins(x=0.04)

        ax.tick_params(
            axis="both",
            which="major",
            labelsize=FONT_TICK * scale,
            pad=6 * scale,
            length=TICK_LEN_MAJOR * scale,
            width=TICK_WIDTH_MAJOR * scale,
        )
        ax.tick_params(
            axis="both",
            which="minor",
            length=TICK_LEN_MINOR * scale,
            width=TICK_WIDTH_MINOR * scale,
        )
        ax.set_xlabel("# of server candidates", fontsize=FONT_LABEL * scale, labelpad=4 * scale)
        ax.set_ylabel("Regret Rate [%]", fontsize=FONT_LABEL * scale, labelpad=6 * scale)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.53, 0.980),
        borderaxespad=0.22,
        fontsize=FONT_LEGEND * scale * 0.82,
        handlelength=3.2,
        handletextpad=0.62,
        columnspacing=1.35,
    )

    fig.subplots_adjust(top=0.85, bottom=0.09, left=0.13, right=0.985, hspace=0.22)

    out_pdf = Path(out_pdf)
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.08)
    print(f"Saved ({uncertainty}) -> {out_pdf.resolve()}")
    plt.show()


# UW only (the many-elements version is removed)
for uncertainty in ["errorbar"]:
    suffix = "errorbar"

    draw_two_panel(
        selectors=["U", "W"],
        out_pdf=f"classic_highlow_2panel_UW_{suffix}.pdf",
        yscale="linear",
        uncertainty=uncertainty,
    )

    draw_two_panel(
        selectors=["U", "W"],
        out_pdf=f"classic_highlow_2panel_UW_linear_{suffix}.pdf",
        yscale="linear",
        uncertainty=uncertainty,
    )

    draw_two_panel_vertical(
        selectors=["U", "W"],
        out_pdf=f"classic_highlow_2panel_UW_vertical_{suffix}.pdf",
        yscale="linear",
        uncertainty=uncertainty,
    )

    draw_two_panel_vertical(
        selectors=["U", "W"],
        out_pdf=f"classic_highlow_2panel_UW_vertical_linear_{suffix}.pdf",
        yscale="linear",
        uncertainty=uncertainty,
    )

    draw_two_panel_vertical_by_scenario(
        selectors=["U", "W"],
        out_pdf=f"classic_highlow_2panel_UW_vertical_by_highlow_{suffix}.pdf",
        yscale="linear",
        uncertainty=uncertainty,
    )





In [ ]:
# === Classic high/low split export (from fig6 last panel; errorbar + no-uncertainty) === fig6 split
# Run this cell after the fig6 plot cell above.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

required_vars = [
    'plot_df', 'SCENARIO_ORDER', 'SHOT_ORDER', 'FLOOR_MIN', 'LINE_WIDTH',
    'FONT_LABEL', 'FONT_TICK', 'FONT_LEGEND', 'TICK_LEN_MAJOR',
    'TICK_LEN_MINOR', 'TICK_WIDTH_MAJOR', 'TICK_WIDTH_MINOR',
    'plot_family_by_shots', 'compute_ymax',
    'SELECTOR_META', 'SHOT_STYLE', 'SHOT_SELECTOR_COLOR', 'clip_pos', 'ENDPOINT_MARKER',
]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        'Run the fig6 plot cell first. Missing variables: ' + ', '.join(missing)
    )

FIGSIZE_SINGLE = (14.0, 8.9)
AXES_BOX = [0.13, 0.14, 0.865, 0.80]
SCALE = 1.3
SPLIT_TEXT_SCALE_UP = 1.1
LEGEND_HANDLE_LENGTH_SPLIT = 3.2 * 0.75
SELECTORS_SPLIT = ['U', 'W']
YMAX_FIG6_SPLIT_HIGH = 50.0  # high-jitter panel fixed at 50%

SCENARIO_SUFFIX = {
    'j2p9m': 'top',
    'j72k': 'bottom',
}


def plot_family_by_shots_no_uncertainty(
    ax,
    data,
    selectors,
    panel_idx,
    line_width=LINE_WIDTH,
    marker_size=10.0,
    marker_edge_width=1.3,
):
    for shots in SHOT_ORDER:
        for sel in selectors:
            d = data[(data['shots'] == int(shots)) & (data['selector'] == sel)].sort_values('group_size')
            if d.empty:
                continue

            x = d['group_size'].to_numpy()
            y = clip_pos(d['mean'].to_numpy())

            meta = SELECTOR_META[sel]
            sh = SHOT_STYLE[int(shots)]
            line_color = SHOT_SELECTOR_COLOR.get(int(shots), {}).get(sel, meta['color'])
            label = f"{meta['name']} ({sh['legend']})" if panel_idx == 0 else None

            ax.plot(
                x,
                y,
                color=line_color,
                linestyle=sh['ls'],
                linewidth=line_width,
                marker=meta['mk'] if ENDPOINT_MARKER else None,
                markevery=[-1] if ENDPOINT_MARKER else None,
                markersize=marker_size,
                markeredgewidth=marker_edge_width,
                label=label,
                zorder=3,
            )


def draw_fig6_split_by_scenario(selectors, out_prefix='fig6_highlow_split', uncertainty='errorbar', out_suffix=''):
    if uncertainty not in {'errorbar', 'none'}:
        raise ValueError(f"uncertainty must be 'errorbar' or 'none', got: {uncertainty}")

    for i, scen in enumerate(SCENARIO_ORDER):
        fig = plt.figure(figsize=FIGSIZE_SINGLE)
        ax = fig.add_axes(AXES_BOX)

        data = plot_df[(plot_df['scenario'] == scen) & (plot_df['selector'].isin(selectors))]
        if uncertainty == 'errorbar':
            plot_family_by_shots(
                ax,
                data,
                selectors,
                panel_idx=0,
                uncertainty='errorbar',
                line_width=LINE_WIDTH * SCALE,
                marker_size=10.0 * SCALE,
                marker_edge_width=1.3 * SCALE,
            )
        else:
            plot_family_by_shots_no_uncertainty(
                ax,
                data,
                selectors,
                panel_idx=0,
                line_width=LINE_WIDTH * SCALE,
                marker_size=10.0 * SCALE,
                marker_edge_width=1.3 * SCALE,
            )

        local_ymax = compute_ymax(data, selectors)
        ax.set_yscale('linear')
        if scen == 'j2p9m':
            y_top = YMAX_FIG6_SPLIT_HIGH
        else:
            y_top = local_ymax * 1.2
        ax.set_ylim(FLOOR_MIN, y_top)
        ax.grid(False, which='both', axis='y')
        ax.margins(x=0.04)

        ax.tick_params(
            axis='both',
            which='major',
            labelsize=FONT_TICK * SCALE * SPLIT_TEXT_SCALE_UP,
            pad=6 * SCALE,
            length=TICK_LEN_MAJOR * SCALE,
            width=TICK_WIDTH_MAJOR * SCALE,
        )
        ax.tick_params(
            axis='both',
            which='minor',
            length=TICK_LEN_MINOR * SCALE,
            width=TICK_WIDTH_MINOR * SCALE,
        )
        ax.set_xlabel('# of server candidates', fontsize=FONT_LABEL * SCALE * SPLIT_TEXT_SCALE_UP, labelpad=4 * SCALE)
        ax.set_ylabel('Regret Rate [%]', fontsize=FONT_LABEL * SCALE * SPLIT_TEXT_SCALE_UP, labelpad=6 * SCALE)

        handles, labels = ax.get_legend_handles_labels()
        leg = ax.legend(
            handles,
            labels,
            loc='upper center',
            bbox_to_anchor=(0.5, 0.995),
            borderaxespad=0.18,
            borderpad=0.24,
            ncol=2,
            frameon=True,
            fontsize=FONT_LEGEND * SCALE * 0.82 * SPLIT_TEXT_SCALE_UP,
            handlelength=LEGEND_HANDLE_LENGTH_SPLIT,
            handletextpad=0.62,
            labelspacing=0.28,
            columnspacing=1.3,
        )
        leg.get_frame().set_edgecolor('black')
        leg.get_frame().set_linewidth(0.8)
        leg.get_frame().set_alpha(1.0)

        out_pdf = Path(f"{out_prefix}_{SCENARIO_SUFFIX.get(scen, scen)}{out_suffix}.pdf")
        fig.savefig(out_pdf, bbox_inches='tight', pad_inches=0.08)
        print(f'Saved ({scen}, {uncertainty}) -> {out_pdf.resolve()}')
        plt.show()


draw_fig6_split_by_scenario(SELECTORS_SPLIT, uncertainty='errorbar', out_suffix='')
draw_fig6_split_by_scenario(SELECTORS_SPLIT, uncertainty='none', out_suffix='_no_uncertainty')


In [ ]:
# === experiment2: 3-parameter sweep (Texe tuned style, enlarged vertical + 2x text/ticks) === fig.3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from pathlib import Path

# Style
plt.rcdefaults()
SIZE_SCALE = 1.4
HEIGHT_STRETCH = 1.7
TEXT_SCALE = 1.10 * SIZE_SCALE
AXIS_SCALE = 1.1  # fig.3: axis label/tick label/tick size scale
AXIS_TITLE_SCALE = 1.03 * 1.03
BASE_FONT = 20
TICK_FONT = BASE_FONT * 1.25 * TEXT_SCALE * AXIS_SCALE
LEGEND_SCALE = 1.10
LEGEND_FONT = (BASE_FONT + 2) * LEGEND_SCALE * TEXT_SCALE

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "font.size": BASE_FONT * TEXT_SCALE,
    "axes.labelsize": (BASE_FONT + 4) * 1.05 * AXIS_TITLE_SCALE * TEXT_SCALE * AXIS_SCALE,
    "axes.titlesize": BASE_FONT * TEXT_SCALE,
    "legend.fontsize": LEGEND_FONT,
    "xtick.labelsize": TICK_FONT,
    "ytick.labelsize": TICK_FONT,
})

# Input
CSV_SWEEP_PATH = Path("../../data/density_sweep2.csv")
if not CSV_SWEEP_PATH.exists():
    candidates = [
        Path("density_sweep2.csv"),
        Path("density_sweep.csv"),
        Path("../data/density_sweep2.csv"),
        Path("../data/density_sweep.csv"),
        Path("../../data/density_sweep.csv"),
    ]
    for cand in candidates:
        if cand.exists():
            CSV_SWEEP_PATH = cand
            break
    else:
        raise FileNotFoundError("density_sweep2.csv / density_sweep.csv not found")

print("Using dataset:", CSV_SWEEP_PATH)
df = pd.read_csv(CSV_SWEEP_PATH)

# Settings
TEXE_COL = "total_time"
NS_TO_MS = 1e6
PER_SHOT = False
SMOOTH_WINDOW = 5
SMOOTH_METHOD = "median"
USE_LOG_X = True
USE_LOG_Y = True

# Keep one figure per parameter; make the figure ~1.7x taller
# Fixed plot-area box so every exported panel has identical graph size
AXES_BOX = [0.20, 0.14, 0.77, 0.82]
BASE_PANEL_HEIGHT = 2.9
FIG_HEIGHT_SINGLE = BASE_PANEL_HEIGHT * 2.0 * HEIGHT_STRETCH
# Use a midpoint width between the original wide layout and square-plot layout.
SQUARE_FIG_WIDTH_SINGLE = FIG_HEIGHT_SINGLE * (AXES_BOX[3] / AXES_BOX[2])
FIG_WIDTH_SINGLE = 0.5 * (12.5 + SQUARE_FIG_WIDTH_SINGLE)
FIGSIZE_SINGLE = (FIG_WIDTH_SINGLE, FIG_HEIGHT_SINGLE)

anchors = ["light", "heavy"]
COLOR_LIGHT = "#377EB8"
COLOR_HEAVY = "#E41A1C"
anchor_styles = {
    "light": dict(color=COLOR_LIGHT, linestyle="-", linewidth=3.0 * SIZE_SCALE),
    "heavy": dict(color=COLOR_HEAVY, linestyle="--", linewidth=3.0 * SIZE_SCALE),
}
anchor_labels = {
    "light": "Light configuration",
    "heavy": "Heavy configuration",
}

x_specs = [
    ("entanglement_speed_factor", r"$R$ [Hz]", "R"),
    ("distance", r"$d$ [km]", "d"),
    ("gate_speed_factor", r"$\kappa$", "kappa"),
]


def texe_ms(frame: pd.DataFrame) -> np.ndarray:
    if TEXE_COL not in frame.columns:
        raise KeyError(f"{TEXE_COL} not found in CSV")
    y = frame[TEXE_COL].astype(float).to_numpy() / NS_TO_MS
    if PER_SHOT:
        if "shots" not in frame.columns:
            raise KeyError("shots column not found (PER_SHOT=True requires shots)")
        y = y / frame["shots"].astype(float).to_numpy()
    return y


def smooth_series(y: np.ndarray, window: int, method: str) -> np.ndarray:
    if window <= 1:
        return y
    s = pd.Series(y)
    if method == "median":
        return s.rolling(window, center=True, min_periods=1).median().to_numpy()
    if method == "mean":
        return s.rolling(window, center=True, min_periods=1).mean().to_numpy()
    raise ValueError("SMOOTH_METHOD must be 'median' or 'mean'")


def summarize_by_x(sub: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for xval, g in sub.groupby("sweep_value", sort=True):
        y = texe_ms(g)
        rows.append({"x": float(xval), "y_mean": float(np.mean(y))})
    out = pd.DataFrame(rows).sort_values("x")
    out["y_smooth"] = smooth_series(out["y_mean"].to_numpy(), SMOOTH_WINDOW, SMOOTH_METHOD)
    return out


def make_dense_log_ticks(ymin: float, ymax: float):
    dmin = int(np.floor(np.log10(ymin)))
    dmax = int(np.ceil(np.log10(ymax)))
    ticks = []
    for d in range(dmin, dmax + 1):
        base = 10 ** d
        for m in (1, 2, 5):
            v = m * base
            if ymin <= v <= ymax:
                ticks.append(v)
    if not ticks:
        ticks = [ymin, ymax]
    return sorted(set(ticks))


def collect_y_range(d: pd.DataFrame):
    y_all = []
    for param_name, _, _ in x_specs:
        for a in anchors:
            sub = d[(d["experiment_anchor"] == a) & (d["experiment_sweep_param"] == param_name)].copy()
            if sub.empty:
                continue
            s = summarize_by_x(sub)
            if not s.empty:
                y_all.extend(s["y_smooth"].astype(float).tolist())

    if not y_all:
        raise RuntimeError("No sweep rows found for experiment2 plot")

    y_min = float(np.min(y_all))
    y_max = float(np.max(y_all))
    if USE_LOG_Y:
        y_min = min(v for v in y_all if v > 0)
        y_lim = (y_min * 0.9, y_max * 1.1)
        y_ticks = make_dense_log_ticks(*y_lim)
    else:
        y_margin = 0.05 * (y_max - y_min)
        y_lim = (y_min - y_margin, y_max + y_margin)
        y_ticks = None
    return y_lim, y_ticks


def draw_one_param(d: pd.DataFrame, param_name: str, xlabel: str, short_name: str, y_lim, y_ticks):
    fig = plt.figure(figsize=FIGSIZE_SINGLE)
    ax = fig.add_axes(AXES_BOX)

    for a in anchors:
        sub = d[(d["experiment_anchor"] == a) & (d["experiment_sweep_param"] == param_name)].copy()
        if sub.empty:
            continue
        s = summarize_by_x(sub)
        ax.plot(s["x"], s["y_smooth"], label=anchor_labels[a], **anchor_styles[a])

    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$T_{\mathrm{exe}}$ [ms]" + (" / shot" if PER_SHOT else ""), labelpad=1.5)

    if USE_LOG_X:
        ax.set_xscale("log")
        ax.xaxis.set_major_locator(mticker.LogLocator(base=10.0))
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))

    if USE_LOG_Y:
        ax.set_yscale("log")
        ax.set_yticks(y_ticks)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=(3, 4, 6, 7, 8, 9)))
        ax.yaxis.set_minor_formatter(mticker.NullFormatter())

    ax.set_ylim(*y_lim)

    # Coarser reference/grid lines (major ticks only)
    ax.grid(True, which="major", linestyle=":", linewidth=0.5)

    # Keep major/minor tick sizes close
    ax.tick_params(axis="both", which="major", labelsize=TICK_FONT, length=10 * SIZE_SCALE * AXIS_SCALE, width=1.6 * SIZE_SCALE * AXIS_SCALE)
    ax.tick_params(axis="y", which="major", pad=1)
    ax.tick_params(axis="both", which="minor", length=10 * SIZE_SCALE * AXIS_SCALE, width=1.2 * SIZE_SCALE * AXIS_SCALE)

    if param_name == "entanglement_speed_factor":
        leg = ax.legend(frameon=True, loc="upper right", bbox_to_anchor=(1.01, 0.98), fontsize=LEGEND_FONT)
        leg.get_frame().set_edgecolor("black")
        leg.get_frame().set_linewidth(0.4)
        leg.get_frame().set_alpha(1.0)

    # Layout is fixed by AXES_BOX to keep all graph areas identical.
    out_pdf = CSV_SWEEP_PATH.parent / f"density_sweep2_texe_{short_name}_v.pdf"
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.08)
    print("Saved:", out_pdf)
    plt.show()


def plot_each_tuned():
    d = df[df["experiment_kind"] == "sweep"].copy()
    y_lim, y_ticks = collect_y_range(d)

    for param_name, xlabel, short_name in x_specs:
        draw_one_param(d, param_name, xlabel, short_name, y_lim, y_ticks)


plot_each_tuned()








In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# fig7
def plot_merged_fig7(
    out_path="fig7_merged.pdf",
    second_out_path=None,
    v_fiber_m_s=2e8,
    d_range=(1e1, 1e4),
    R_range=(1e1, 1e4),
    even_box=((100, 1000), (200, 2000)),  # (dmin,dmax),(Rmin,Rmax)
    U_list=(1, 5, 20),
    show_sobol_box=False,
    show_kappa_panel=True,
    kappa=1.0,
    kappa_d_coeff_km=379.0,
    kappa_R_coeff_hz=527.0,
    figsize=(7.2, 5.8),
    dpi=180,
):
    """
    Merged Fig.7:
      1) Operating map + multi-user boundary shifts (original)
      2) Same map + Sec. operating-map approximation guides:
         d = 379*kappa [km], R = 527/kappa [Hz]
    Assumes PB-VQE with K=n_cc -> R_bal(d)=v_fiber/d (Hz, d in km).
    """
    v_fiber_km_s = v_fiber_m_s / 1000.0  # 2e8 m/s -> 2e5 km/s

    dmin, dmax = d_range
    Rmin, Rmax = R_range
    d = np.logspace(np.log10(dmin), np.log10(dmax), 400)

    if second_out_path is None:
        p = Path(out_path)
        second_out_path = str(p.with_name(f"{p.stem}_kappa{p.suffix}"))

    # Keep this figure stable even when previous cells changed global rcParams.
    TICK_MARK_SCALE = 2.0
    TICK_LABEL_SCALE = 1.45
    LEGEND_TEXT_SCALE = 1.3 * 1.2
    rc_local = {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "mathtext.fontset": "dejavusans",
        "font.size": 12,
        "axes.labelsize": 18,
        "xtick.labelsize": 12 * TICK_LABEL_SCALE,
        "ytick.labelsize": 12 * TICK_LABEL_SCALE,
        "legend.fontsize": 11 * LEGEND_TEXT_SCALE,
    }

    boundary_styles = {
        1: {"color": "#377EB8", "ls": "-"},
        5: {"color": "#984EA3", "ls": "-"},
        20: {"color": "#4DAF4A", "ls": "-"},
    }
    color_even = "#E41A1C"
    color_sobol = "#9E9E9E"

    def _draw_base_map(ax):
        # --- Boundary lines: T_cc = T_ent, shifted by U (contention model R_eff = R/U) ---
        for U in U_list:
            st = boundary_styles.get(U, {"color": "#9E9E9E", "ls": "-"})
            R_bal_U = (U * v_fiber_km_s) / d
            ax.loglog(
                d,
                R_bal_U,
                linestyle=st["ls"],
                color=st["color"],
                linewidth=2.4,
                label=fr"$U={U}$: $T_{{cc}}=T_{{ent}}$",
            )

        # --- Even regime box (Table 1) ---
        (d1, d2), (R1, R2) = even_box
        ax.plot(
            [d1, d2, d2, d1, d1],
            [R1, R1, R2, R2, R1],
            linestyle="-",
            color=color_even,
            linewidth=2.4,
            label="Even regime",
        )

        # --- Sobol prior box (Table 3) ---
        if show_sobol_box:
            ax.plot(
                [dmin, dmax, dmax, dmin, dmin],
                [Rmin, Rmin, Rmax, Rmax, Rmin],
                linestyle="--",
                color=color_sobol,
                linewidth=2.4,
                label="Sobol prior (Table 3)",
            )

    def _style_axes(ax):
        ax.set_xlabel(r"Channel length $d$ [km]", labelpad=3)
        ax.set_ylabel(r"Entanglement distribution rate $R$ [Hz]", labelpad=3)
        ax.set_xlim(dmin, dmax)
        ax.set_ylim(Rmin, Rmax)

        major_len = plt.rcParams.get("xtick.major.size", 3.5) * TICK_MARK_SCALE
        major_w = plt.rcParams.get("xtick.major.width", 0.8) * TICK_MARK_SCALE
        minor_len = plt.rcParams.get("xtick.minor.size", 2.0) * TICK_MARK_SCALE
        minor_w = plt.rcParams.get("xtick.minor.width", 0.6) * TICK_MARK_SCALE
        ax.tick_params(axis="both", which="major", length=major_len, width=major_w)
        ax.tick_params(axis="both", which="minor", length=minor_len, width=minor_w)

        ax.grid(True, which="major", linestyle=":", linewidth=0.5)

    def _legend_top_2col(ax):
        ax.legend(
            loc="lower left",
            bbox_to_anchor=(0.01, 0.01),
            ncol=1,
            framealpha=0.99,
            columnspacing=1.0,
            handlelength=2.4,
        )

    # --- Panel 1: original map (kept as-is) ---
    with plt.rc_context(rc_local):
        fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
        _draw_base_map(ax)
        _style_axes(ax)
        _legend_top_2col(ax)
        fig.savefig(out_path, bbox_inches="tight", dpi=dpi)
        plt.show()
    print(f"Saved: {out_path}")

    # --- Panel 2: add Sec. operating-map approximation lines (no filled region) ---
    if show_kappa_panel:
        kappa = float(kappa)
        if kappa <= 0:
            raise ValueError("kappa must be > 0")

        d_cap = kappa_d_coeff_km * kappa
        R_floor = kappa_R_coeff_hz / kappa

        with plt.rc_context(rc_local):
            fig2, ax2 = plt.subplots(figsize=figsize, constrained_layout=True)
            _draw_base_map(ax2)

            guide_color = "#111111"
            guide_label_scale = 1.05
            ax2.vlines(
                d_cap,
                ymin=Rmin,
                ymax=R_floor,
                color=guide_color,
                linestyle="--",
                linewidth=2.0,
                alpha=0.95,
                zorder=4,
            )
            ax2.hlines(
                R_floor,
                xmin=dmin,
                xmax=d_cap,
                color=guide_color,
                linestyle="--",
                linewidth=2.0,
                alpha=0.95,
                zorder=4,
            )
            if (dmin <= d_cap <= dmax) and (Rmin <= R_floor <= Rmax):
                ax2.scatter(
                    [d_cap],
                    [R_floor],
                    marker="o",
                    color=guide_color,
                    s=90,
                    linewidths=0.8,
                    zorder=6,
                )
                ax2.annotate(
                    fr"$d={d_cap:.0f}$",
                    xy=(d_cap, Rmin),
                    xytext=(6, 6),
                    textcoords="offset points",
                    fontsize=plt.rcParams.get("legend.fontsize", 11) * guide_label_scale,
                    color=guide_color,
                    ha="left",
                    va="bottom",
                    zorder=6,
                )
                ax2.annotate(
                    fr"$R={R_floor:.0f}$",
                    xy=(dmin, R_floor),
                    xytext=(6, 2),
                    textcoords="offset points",
                    fontsize=plt.rcParams.get("legend.fontsize", 11) * guide_label_scale,
                    color=guide_color,
                    ha="left",
                    va="bottom",
                    zorder=6,
                )

            _style_axes(ax2)
            _legend_top_2col(ax2)
            fig2.savefig(second_out_path, bbox_inches="tight", dpi=dpi)
            plt.show()
        print(f"Saved: {second_out_path}")


plot_merged_fig7(out_path="fig7_merged.pdf")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from pathlib import Path

# ===== Fig_validity (Tmax vs Texe scatter) ===== fig10
plt.rcdefaults()
BASE_FONT = 19
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "font.size": BASE_FONT,
    "axes.labelsize": int((BASE_FONT + 1) * 1.0),
    "axes.titlesize": BASE_FONT,
    "axes.linewidth": 1.8,


    "xtick.labelsize": BASE_FONT,
    "ytick.labelsize": BASE_FONT,
})

# Input settings
# CSV_EVEN_PATH = Path("../../data/density_random_custom_client_gate_speed_factor-1-1_distance-100-10000_entanglement_speed_factor-200-2000_gate_speed_factor-0_263-2_63_num_runs-1_shots-10_samples-10000_sets-1_seed-42_stride-997_paramseed-42.csv")
CSV_EVEN_PATH = Path("../../data/dr_even_base10000_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv")

CSV_CANDIDATES = [
    Path("../../data/dr_even_base10000_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-mixed_es-0.csv"),
    Path("../../data/dr_ext_10_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-42.csv"),
    Path("../data/dr_ext_10_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-42.csv"),
    Path("dr_ext_10_jit-0_cgsf-1_dist-100-1000_esf-200-2000_gsf-0_263-2_63_nr-1_sh-10_smp-10000_set-1_sd-mixed_str-997_psd-42.csv"),
    Path("density_random_custom_client_gate_speed_factor-1-1_distance-100-1000_entanglement_speed_factor-200-2000_gate_speed_factor-0_263-2_63_num_runs-1_shots-10_samples-10000_sets-1_seed-42_stride-997_paramseed-42.csv"),
    Path("../data/density_random_custom_client_gate_speed_factor-1-1_distance-100-1000_entanglement_speed_factor-200-2000_gate_speed_factor-0_263-2_63_num_runs-1_shots-10_samples-10000_sets-1_seed-42_stride-997_paramseed-42.csv"),
    Path("../../data/density_random_custom_client_gate_speed_factor-1-1_distance-100-1000_entanglement_speed_factor-200-2000_gate_speed_factor-0_263-2_63_num_runs-1_shots-10_samples-10000_sets-1_seed-42_stride-997_paramseed-42.csv"),
]
SAMPLE_SIZES = [None, 5000, 2000]  # None means all rows
RNG_SEED = 42

for cand in CSV_CANDIDATES:
    if cand.exists():
        CSV_EVEN_PATH = cand
        break
else:
    raise FileNotFoundError("Even-regime CSV not found (checked candidates)")
print("Using dataset:", CSV_EVEN_PATH)

df = pd.read_csv(CSV_EVEN_PATH)
shots_series = df["shots"].astype(float)
shots_value = float(shots_series.iloc[0]) if len(shots_series) else np.nan
if not np.allclose(shots_series, shots_value):
    print("Warning: shots column varies; using row-wise values for scaling.")

CLIENT_MS = 2.35  # per-shot client runtime (shared by ZZ/XX)
XX_SERVER_EXTRA_MS = 0.135  # additional server time for XX (135 µs)

# Server-side times for ZZ and XX
tsrv_zz_ms = 9.48 * df["gate_speed_factor"].astype(float)
tsrv_xx_ms = tsrv_zz_ms + XX_SERVER_EXTRA_MS

# Channel / entanglement times (shared)
tcc_ms = 0.025 * df["distance"].astype(float)
tent_ms = 5000.0 / df["entanglement_speed_factor"].astype(float)

# Additive score: ZZ + XX (both include client time; XX has extra server time)
per_shot_tmax = (
    tsrv_zz_ms + tcc_ms + tent_ms + CLIENT_MS
    + tsrv_xx_ms + tcc_ms + tent_ms + CLIENT_MS
)
tmax_ms = shots_series * per_shot_tmax

# Bottleneck score: max per shot (include client); sum of ZZ-max and XX-max
client_arr = np.full_like(tsrv_zz_ms, CLIENT_MS)
per_shot_tmin_zz = np.maximum.reduce([tsrv_zz_ms, tcc_ms, tent_ms, client_arr])
per_shot_tmin_xx = np.maximum.reduce([tsrv_xx_ms, tcc_ms, tent_ms, client_arr])
per_shot_tmin = per_shot_tmin_zz + per_shot_tmin_xx
tmin_ms = shots_series * per_shot_tmin
texe_ms = df["total_time"].astype(float) / 1e6  # ns -> ms

def spearman_rho_safe(x_arr, y_arr):
    try:
        from scipy.stats import spearmanr
        rho_val, _ = spearmanr(x_arr, y_arr)
        return float(rho_val)
    except Exception:
        try:
            return float(pd.Series(x_arr).rank().corr(pd.Series(y_arr).rank()))
        except Exception:
            return float("nan")

for sample_size in SAMPLE_SIZES:
    n_total = len(df)
    if sample_size is None:
        n_pick = n_total
        indices = np.arange(n_total)
        suffix = ""
        sample_label = "all"
    else:
        n_pick = int(min(sample_size, n_total))
        rng = np.random.default_rng(RNG_SEED)
        indices = rng.choice(n_total, size=n_pick, replace=False)
        suffix = f"_n{n_pick}"
        sample_label = f"{n_pick}"

    x_tmax = tmax_ms.iloc[indices].to_numpy()
    x_tmin = tmin_ms.iloc[indices].to_numpy()
    y = texe_ms.iloc[indices].to_numpy()

    rho_tmax = spearman_rho_safe(x_tmax, y)
    rho_tmin = spearman_rho_safe(x_tmin, y)
    print(f"Spearman rho (sample={sample_label}, seed={RNG_SEED}): T_max={rho_tmax:.4f}, T_min={rho_tmin:.4f}")

    fig, ax = plt.subplots(figsize=(8.2, 6.6 * 0.8))
    COLOR_TMAX = "#377EB8"
    COLOR_TMIN = "#4D4D4D"

    scatter_tmax = ax.scatter(
        x_tmax,
        y,
        s=18,
        color=COLOR_TMAX,
        alpha=0.25,
        linewidth=0,
        marker="o",
        label=r"$T_{\mathrm{max}}$",
    )
    scatter_tmin = ax.scatter(
        x_tmin,
        y,
        s=18,
        color=COLOR_TMIN,
        alpha=0.25,
        linewidth=0,
        marker="o",
        label=r"$T_{\mathrm{min}}$",
    )

    line_min = min(x_tmax.min(), x_tmin.min(), y.min())
    line_max = max(x_tmax.max(), x_tmin.max(), y.max())
    ax.plot(
        [line_min, line_max],
        [line_min, line_max],
        linestyle="--",
        linewidth=2.2,
        color="#9E9E9E",
        alpha=0.8,
        label="y = x",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10.0))
    ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0))
    ax.xaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.yaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.tick_params(which="both", width=1.8, length=15)
    ax.tick_params(which="minor", length=10)

    ax.set_xlabel(r"Estimated runtime [ms]", labelpad=2)
    ax.set_ylabel(r"Simulated runtime $T_{\mathrm{exe}}$ [ms]", labelpad=2)

    legend_fontsize = (BASE_FONT - 2) * 1.2
    leg = ax.legend(frameon=True, loc="lower right", fontsize=legend_fontsize, markerscale=2.0)
    legend_handles = getattr(leg, "legend_handles", getattr(leg, "legendHandles", []))
    for h in legend_handles:
        if hasattr(h, "set_alpha"):
            h.set_alpha(1.0)
    leg.get_frame().set_edgecolor("black")
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_alpha(1.0)
    ax.grid(False)
    fig.subplots_adjust(top=0.95, bottom=0.16, left=0.18, right=0.98)

    out_pdf = CSV_EVEN_PATH.parent / f"{CSV_EVEN_PATH.stem}_fig_validity{suffix}.pdf"
    fig.savefig(out_pdf, bbox_inches="tight")
    print("Saved figure:", out_pdf)

    rho_log = CSV_EVEN_PATH.parent / f"{CSV_EVEN_PATH.stem}_fig_validity_rho{suffix}.txt"
    rho_log.write_text(f"seed={RNG_SEED}, N={n_pick}, rho_tmax={rho_tmax:.6f}, rho_tmin={rho_tmin:.6f}\n")
    print("Logged rho:", rho_log)

    plt.show()


In [ ]:
# === time_density single CSV visualization (self-contained) ===
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_NAME = "time_density_dist-1e1-1e4x6_ent-2e1-2e4x6_gs-3e-2-3e1x6_N128_shots10.csv"
CSV_CANDIDATES = [
    Path("../../data") / CSV_NAME,
    Path("../data") / CSV_NAME,
    Path("data") / CSV_NAME,
    Path("_public_snapshot/data") / CSV_NAME,
    Path("pb-vqe2/data") / CSV_NAME,
]
for cand in CSV_CANDIDATES:
    if cand.exists():
        input_file = cand
        break
else:
    raise FileNotFoundError(f"CSV not found: {CSV_NAME}")


def _fmt_tick_value(value, sig=6):
    try:
        value = float(value)
    except Exception:
        return str(value)
    if not np.isfinite(value):
        return ""
    s = f"{value:.{sig}g}"
    return s.replace("e+0", "e").replace("e+", "e").replace("e-0", "e-")


def visualize_3d_segments(
    csv_path,
    param_names=None,
    target_metric_name=None,
    output_file=None,
    invert_x_axis=False,
    invert_y_axis=False,
    use_log_scale=None,
    equalize_log_spacing=True,
    output_dpi=600,
    export_vector=True,
    rasterize_points=True,
    fig_size=None,
    points_alpha=1.0,
):
    data = pd.read_csv(csv_path)

    if param_names is None:
        contribution_columns = [col for col in data.columns if col.endswith("_contribution")]
        param_names = [col.replace("_contribution", "") for col in contribution_columns[:3]]
        if len(param_names) != 3:
            raise ValueError("Could not infer exactly three parameters; pass param_names explicitly")

    if target_metric_name is None or target_metric_name not in data.columns:
        for metric in ("target_metric", "total_time"):
            if metric in data.columns:
                target_metric_name = metric
                break
        if target_metric_name is None:
            numeric_cols = [c for c in data.columns if pd.api.types.is_numeric_dtype(data[c])]
            if not numeric_cols:
                raise ValueError("No numeric column available for target metric")
            target_metric_name = numeric_cols[-1]

    if use_log_scale is None:
        if "use_log_scale" in data.columns:
            raw = data["use_log_scale"].astype(str).str.lower().str.strip()
            use_log_scale = raw.isin(["true", "1", "yes", "y", "t"]).any()
        else:
            use_log_scale = False

    print(f"Parameters used: {param_names}")
    print(f"Target metric used: {target_metric_name}")
    print(f"Scale used: {'logarithmic' if use_log_scale else 'linear'}")

    param_data = {}
    unique_values = {}
    for param in param_names:
        series = pd.to_numeric(data[param], errors="raise")
        if use_log_scale:
            safe = series.copy()
            if (safe <= 0).any():
                min_positive = safe[safe > 0].min()
                if pd.isna(min_positive):
                    raise ValueError(f"{param} has no positive values for log scale")
                safe = safe + min_positive * 0.1
            uniq = np.array(sorted(safe.unique()))
            unique_values[param] = uniq
            if equalize_log_spacing:
                index_map = {v: i for i, v in enumerate(uniq)}
                param_data[param] = safe.map(index_map).astype(float)
            else:
                param_data[param] = np.log10(safe)
        else:
            uniq = np.array(sorted(series.unique()))
            unique_values[param] = uniq
            param_data[param] = series.astype(float)

    metric = pd.to_numeric(data[target_metric_name], errors="raise")
    min_metric = float(metric.min())
    max_metric = float(metric.max())
    if max_metric == min_metric:
        normalized_sizes = np.full(len(data), 2 * 260.0)
    else:
        normalized_sizes = 2 * (260 + (metric - min_metric) / (max_metric - min_metric) * 2600)

    contribution_columns = [f"{param}_contribution" for param in param_names]
    if all(col in data.columns for col in contribution_columns):
        dominant_factor = data[contribution_columns].idxmax(axis=1).str.replace("_contribution", "", regex=False)
    else:
        dominant_factor = pd.Series([param_names[0]] * len(data), index=data.index)

    axis_label_map = {
        "distance": "d[km]",
        "entanglement_speed_factor": "R[Hz]",
        "gate_speed_factor": r"$\kappa$",
        "gate_time_ns": "Operation duration (us)",
        "coherence_time": "T1 relaxation time (s)",
        "entanglement_fidelity": "Entanglement fidelity",
        "noise_rate": "Operation-fidelity scaling factor",
        "operation_fidelity": "Operation fidelity",
    }

    color_palette = ["#CC3366", "#FFCC00", "#3366CC"]
    colors = {p: color_palette[i % len(color_palette)] for i, p in enumerate(param_names)}
    labels = {p: axis_label_map.get(p, p) for p in param_names}

    fig = plt.figure(figsize=(fig_size or (22, 18)), constrained_layout=False)
    ax = fig.add_subplot(111, projection="3d")

    for param in param_names:
        mask = dominant_factor == param
        if not mask.any():
            continue
        ax.scatter(
            param_data[param_names[0]][mask].values,
            param_data[param_names[1]][mask].values,
            param_data[param_names[2]][mask].values,
            s=normalized_sizes[mask].values,
            c=colors[param],
            alpha=float(points_alpha),
            edgecolors="black",
            label=labels[param],
            rasterized=bool(rasterize_points),
        )

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=colors[p],
            markeredgecolor="black",
            markersize=26,
            alpha=float(points_alpha),
        )
        for p in param_names
    ]
    legend_labels = [labels[p] for p in param_names]
    fig.legend(
        legend_handles,
        legend_labels,
        title="Dominant Factor",
        loc="upper center",
        bbox_to_anchor=(0.5, 0.88),
        ncol=len(legend_labels),
        frameon=True,
        fontsize=30,
        title_fontsize=38,
    )
    fig.subplots_adjust(top=0.82)

    if invert_x_axis:
        ax.invert_xaxis()
    if invert_y_axis:
        ax.invert_yaxis()

    ax.set_xlabel(labels[param_names[0]], fontsize=28, labelpad=30)
    ax.set_ylabel(labels[param_names[1]], fontsize=28, labelpad=35)
    ax.set_zlabel(labels[param_names[2]], fontsize=28, labelpad=40)
    ax.grid(True, alpha=0.25)

    for axis_index, param in enumerate(param_names):
        raw_ticks = unique_values[param]
        if len(raw_ticks) > 6:
            step = max(1, len(raw_ticks) // 6)
            raw_ticks = raw_ticks[::step]

        if use_log_scale and equalize_log_spacing:
            full = unique_values[param]
            index_map = {v: i for i, v in enumerate(full)}
            tick_pos = np.array([index_map[v] for v in raw_ticks], dtype=float)
        elif use_log_scale:
            tick_pos = np.log10(raw_ticks)
        else:
            tick_pos = raw_ticks

        tick_labels = [_fmt_tick_value(v) for v in raw_ticks]

        if axis_index == 0:
            ax.set_xticks(tick_pos)
            ax.set_xticklabels(tick_labels)
        elif axis_index == 1:
            ax.set_yticks(tick_pos)
            ax.set_yticklabels(tick_labels)
        else:
            ax.set_zticks(tick_pos)
            ax.set_zticklabels(tick_labels)

    if output_file:
        output_path = Path(output_file)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=output_dpi, bbox_inches="tight")
        if export_vector:
            fig.savefig(output_path.with_suffix(".pdf"), bbox_inches="tight")
            fig.savefig(output_path.with_suffix(".svg"), bbox_inches="tight")

    return fig


preview = pd.read_csv(input_file, nrows=8)
third = "gate_time_ns" if "gate_time_ns" in preview.columns else "gate_speed_factor"
param_names = ["distance", "entanglement_speed_factor", third]
use_log = False
if "use_log_scale" in preview.columns:
    raw = preview["use_log_scale"].astype(str).str.lower().str.strip()
    use_log = raw.isin(["true", "1", "yes", "y", "t"]).any()

out_dir = input_file.parent / "time_density_vis"
out_dir.mkdir(parents=True, exist_ok=True)
out_pdf = out_dir / f"{input_file.stem}_3d.pdf"
out_svg = out_dir / f"{input_file.stem}_3d.svg"

_orig_show = plt.show
plt.show = lambda *args, **kwargs: None
try:
    fig = visualize_3d_segments(
        str(input_file),
        param_names=param_names,
        target_metric_name="target_metric",
        output_file=None,
        invert_x_axis=True,
        invert_y_axis=False,
        use_log_scale=use_log,
        equalize_log_spacing=False,
        output_dpi=800,
        export_vector=False,
    )
finally:
    plt.show = _orig_show

if fig is None or len(getattr(fig, "axes", [])) == 0:
    raise RuntimeError("Failed to get a valid 3D figure")
ax = fig.axes[0]

z_label = r"$\kappa$" if third == "gate_speed_factor" else "Operation duration (us)"
ax.set_xlabel("d[km]", fontsize=33, labelpad=40)
ax.set_ylabel("R[Hz]", fontsize=33, labelpad=45)
ax.set_zlabel(z_label, fontsize=33, labelpad=52)
if fig.legends:
    forced_labels = ["d[km]", "R[Hz]", z_label]
    for lg in fig.legends:
        for txt, new_label in zip(lg.get_texts(), forced_labels):
            txt.set_text(new_label)

ticks = {
    "distance": np.array([10, 100, 1000, 10000], dtype=float),
    "entanglement_speed_factor": np.array([20, 200, 2000, 20000], dtype=float),
    "gate_speed_factor": np.array([0.03, 0.3, 3, 30], dtype=float),
}
for axis_index, param in enumerate(param_names):
    if param not in ticks:
        continue
    raw_ticks = ticks[param]
    tick_pos = np.log10(raw_ticks) if use_log else raw_ticks
    labels = [_fmt_tick_value(v) for v in raw_ticks]
    if param == "distance" and labels:
        labels[0] = labels[0] + "  "
    if param == "entanglement_speed_factor" and labels:
        labels[-1] = "  " + labels[-1]
    if axis_index == 0:
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(labels, fontsize=25)
    elif axis_index == 1:
        ax.set_yticks(tick_pos)
        ax.set_yticklabels(labels, fontsize=25)
    else:
        ax.set_zticks(tick_pos)
        ax.set_zticklabels(labels, fontsize=25)

fig.set_size_inches(25, 21)
ax.view_init(elev=20, azim=55)
ax.tick_params(axis="x", which="major", labelsize=30, pad=10)
ax.tick_params(axis="y", which="major", labelsize=30, pad=10)
ax.tick_params(axis="z", which="major", labelsize=30, pad=17)
fig.subplots_adjust(top=0.81, left=0.27, bottom=0.27, right=0.995)
fig.canvas.draw()

fig.savefig(out_pdf, dpi=800, facecolor="white", bbox_inches="tight", pad_inches=0.42)
fig.savefig(out_svg, facecolor="white", bbox_inches="tight", pad_inches=0.42)
print("Input file:", input_file)
print("Saved:", out_pdf)
print("Saved:", out_svg)
plt.show()
